# Day 4 - Pandas Full Workflow Colab Examples

This notebook is a Google Colab friendly worked example of a complete pandas workflow:
`load -> inspect -> clean -> filter -> combine -> summarize -> reshape -> visualize -> export`.

It keeps the same Day 4 theory and the same demo datasets as the local working notebook, but it is now hybrid:
- if local `data/day4/raw/` files exist, it uses them directly;
- otherwise it loads the source files from this GitHub repository where appropriate:
`https://github.com/ValRCS/RTU_Python_CSP`.

Notebook behavior:
- The setup cell detects whether local Day 4 data is available.
- In local mode, inputs are read from the repository and outputs are written into `data/day4/outputs/`.
- In remote mode, CSV, JSON, and HTML inputs are loaded from GitHub raw URLs.
- In remote mode, binary files such as `SQLite` and `Excel` are first downloaded into a runtime cache, then read locally.
- The final cells generate exports in the active output folder.

Important note:
- The raw URLs target the `main` branch of this repository.
- If you update the local `data/day4/raw/` files, push them to GitHub before relying on this Colab notebook.

Recommended usage:
1. Open the notebook in Google Colab.
2. Or run it locally in Jupyter from the repository.
3. Run the notebook from top to bottom.
4. Compare the code to the theory notes and adjust parameters interactively if needed.
5. Inspect the generated files in the active output folder.


## Workflow Map

A strong pandas workflow is iterative rather than perfectly linear, but this order is usually the most reliable:
- `Load`: bring source data into one or more DataFrames with as little accidental distortion as possible.
- `Inspect`: learn the table grain, schema, data types, missingness, duplicates, and likely problem areas.
- `Clean`: standardize names, types, values, categories, and row-level quality issues.
- `Filter`: keep only the rows and columns required for the current question.
- `Combine`: merge related tables or stack repeated extracts.
- `Summarize`: produce grouped metrics, pivot-style reports, and decision-ready aggregates.
- `Reshape`: convert between wide and long layouts depending on reporting and plotting needs.
- `Visualize`: turn prepared tables into charts that answer a specific question.
- `Export`: save cleaned data, summary tables, and outputs in reusable formats.

Useful mental model:
- Early stages protect data fidelity.
- Middle stages create analysis-ready tables.
- Late stages communicate and deliver results.

Also expect to loop back:
- Inspection may show that the data should be loaded with different parameters.
- Cleaning may reveal that filters need to be delayed or revised.
- Combining may expose key mismatches that require more cleaning.
- Visualization often reveals that a summary or reshape step should change.


In [1]:
from __future__ import annotations

# standarta bibliotēkas importi
import importlib.util
import sqlite3
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

# vārdnīca ar nepieciešamajām pakotnēm un to instalēšanas nosaukumiem
REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
    "lxml": "lxml",
    "bs4": "beautifulsoup4",
    "html5lib": "html5lib",
}
missing_packages = sorted(
    {package for module_name, package in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None}
)
# ar subprocess mēs varam izsaukt pip, lai instalētu trūkstošās pakotnes
# vispārīgi ar subproces var izsaukt jebkuru termināļa komandu, bet šeit mēs izmantojam to, lai izsauktu pip instalēšanu
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
# parādam pandas versiju
print(f"Pandas version: {pd.__version__}")

REPO_WEB_URL = "https://github.com/ValRCS/RTU_Python_CSP"
RAW_BASE_URL = "https://raw.githubusercontent.com/ValRCS/RTU_Python_CSP/main/data/day4/raw"

# ar nākošo funkciju mēs meklējam repozitorija sakni, lai varētu izmantot lokālos datus, ja tie ir pieejami
def find_repo_root() -> Path | None:
    candidates: list[Path] = []

    try:
        candidates.append(Path(__file__).resolve().parents[1])
    except NameError:
        pass

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, cwd.parent, cwd.parent.parent if len(cwd.parents) >= 2 else cwd])

    for candidate in candidates:
        if (candidate / "data" / "day4" / "raw").exists() and (candidate / "notebooks").exists():
            return candidate

    return None


LOCAL_REPO_ROOT = find_repo_root()
USE_LOCAL_DATA = LOCAL_REPO_ROOT is not None
IN_COLAB = "google.colab" in sys.modules

if USE_LOCAL_DATA:
    DATA_DIR = LOCAL_REPO_ROOT / "data" / "day4"
    RAW_DIR = DATA_DIR / "raw"
    INTERIM_DIR = DATA_DIR / "interim"
    OUTPUT_DIR = DATA_DIR / "outputs"
    RAW_CACHE_DIR = RAW_DIR
    DATA_SOURCE_MODE = "local"
else:
    RUNTIME_ROOT = Path.cwd() / "day4_colab_runtime"
    RAW_CACHE_DIR = RUNTIME_ROOT / "raw_cache"
    INTERIM_DIR = RUNTIME_ROOT / "interim"
    OUTPUT_DIR = RUNTIME_ROOT / "outputs"
    DATA_SOURCE_MODE = "remote"

for path in (RAW_CACHE_DIR, INTERIM_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.precision", 2)

EXCEL_ENGINE = "openpyxl"

if USE_LOCAL_DATA:
    demo_files = {
        "sales_january_csv": RAW_DIR / "sales_january_raw.csv",
        "sales_february_csv": RAW_DIR / "sales_february_raw.csv",
        "products_json": RAW_DIR / "products_catalog.json",
        "stores_html": RAW_DIR / "stores.html",
        "stores_csv": RAW_DIR / "stores.csv",
        "targets_sqlite": RAW_DIR / "regional_targets.sqlite",
        "reference_excel": RAW_DIR / "reference_tables.xlsx",
    }
else:
    demo_files = {
        "sales_january_csv": f"{RAW_BASE_URL}/sales_january_raw.csv",
        "sales_february_csv": f"{RAW_BASE_URL}/sales_february_raw.csv",
        "products_json": f"{RAW_BASE_URL}/products_catalog.json",
        "stores_html": f"{RAW_BASE_URL}/stores.html",
        "stores_csv": f"{RAW_BASE_URL}/stores.csv",
        "targets_sqlite": f"{RAW_BASE_URL}/regional_targets.sqlite",
        "reference_excel": f"{RAW_BASE_URL}/reference_tables.xlsx",
    }

{
    "data_source_mode": DATA_SOURCE_MODE,
    "in_colab": IN_COLAB,
    "output_dir": OUTPUT_DIR,
    "demo_files": demo_files,
}


Pandas version: 3.0.2


{'data_source_mode': 'local',
 'in_colab': False,
 'output_dir': WindowsPath('D:/Github/RTU_Python_CSP/data/day4/outputs'),
 'demo_files': {'sales_january_csv': WindowsPath('D:/Github/RTU_Python_CSP/data/day4/raw/sales_january_raw.csv'),
  'sales_february_csv': WindowsPath('D:/Github/RTU_Python_CSP/data/day4/raw/sales_february_raw.csv'),
  'products_json': WindowsPath('D:/Github/RTU_Python_CSP/data/day4/raw/products_catalog.json'),
  'stores_html': WindowsPath('D:/Github/RTU_Python_CSP/data/day4/raw/stores.html'),
  'stores_csv': WindowsPath('D:/Github/RTU_Python_CSP/data/day4/raw/stores.csv'),
  'targets_sqlite': WindowsPath('D:/Github/RTU_Python_CSP/data/day4/raw/regional_targets.sqlite'),
  'reference_excel': WindowsPath('D:/Github/RTU_Python_CSP/data/day4/raw/reference_tables.xlsx')}}

In [ ]:
# ierakstot pd.read mēs redzēsim visas pieejamās funkcijas datu ielādei, piemēram, pd.read_csv, pd.read_excel, pd.read_json, pd.read_html utt.

## 1. Load

Goal: bring source data into pandas with minimal accidental transformation.

Loading is not just an import step. It is where you define how pandas should interpret separators, headers, data types, missing values, dates, decimal symbols, indexes, sheet names, query results, and file encodings. Good loading choices reduce downstream cleaning work and preserve important information such as leading zeros, categorical labels, and date precision.

Common source patterns to demonstrate:
- Delimited text with `pd.read_csv()`, `pd.read_table()`, or `pd.read_fwf()`.
- Spreadsheet files with `pd.read_excel()` from one sheet, many sheets, or a sheet dictionary.
- Semi-structured text with `pd.read_json()`, `pd.json_normalize()`, `pd.read_html()`, or `pd.read_xml()`.
- Columnar and binary formats with `pd.read_parquet()`, `pd.read_feather()`, or `pd.read_pickle()`.
- Databases with `pd.read_sql_query()`, `pd.read_sql_table()`, or `pd.read_sql()`.
- Quick manual sources with `pd.read_clipboard()` or a DataFrame created from Python lists, dictionaries, or API responses.

Parameters worth discussing early:
- `usecols`: load only the columns you actually need.
- `dtype`: prevent pandas from guessing incorrectly, especially for IDs and codes.
- `parse_dates`: parse dates at load time when the format is dependable.
- `index_col`: choose a meaningful index only when it helps later steps.
- `nrows` or `chunksize`: useful for previews and large files.
- `na_values` and `keep_default_na`: define what counts as missing.
- `encoding`, `sep`, `decimal`, and `thousands`: essential for locale-sensitive files.
- `sheet_name`: useful for multi-sheet Excel workbooks.
- storage or engine options: especially relevant for Excel, parquet, and remote storage.

Questions to answer while loading:
- What does one row represent?
- Which columns are mandatory for the analysis question?
- Are there columns that must stay as text even if they look numeric?
- Are there multiple source files, monthly extracts, or sheets that should later be combined?
- Is the data already flat, or will nested structures need normalization?
- Should the first pass load all rows, a small preview, or chunks?

Common pitfalls:
- Losing leading zeros in IDs by allowing automatic numeric conversion.
- Treating locale-specific numbers like text because decimal or thousands symbols were ignored.
- Pulling far more columns than needed, which increases memory use and confusion.
- Assuming JSON is already tabular when nested objects or lists still need flattening.
- Forgetting that SQL can be part of the pandas workflow, not a separate universe.

Documentation references:
- [pandas IO tools user guide](https://pandas.pydata.org/docs/user_guide/io.html)
- [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [pandas.read_excel](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html)
- [pandas.read_sql_query](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html)
- [pandas.read_parquet](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html)


In [2]:
def ensure_local_binary(source: str | Path, filename: str) -> Path:
    if isinstance(source, Path):
        return source
    destination = RAW_CACHE_DIR / filename
    urlretrieve(source, destination)
    return destination


january_csv_source = demo_files["sales_january_csv"]
february_csv_source = demo_files["sales_february_csv"]
products_json_source = demo_files["products_json"]
stores_html_source = demo_files["stores_html"]
stores_csv_source = demo_files["stores_csv"]
sqlite_source = demo_files["targets_sqlite"]
excel_source = demo_files["reference_excel"]

# izdrukājam visus datu avotus vēlreiz:
for name, source in demo_files.items():
    print(f"{name}: {source}")

# izmantoju assert lai apstātos pirm datu ielādes brīdī, lai varētu pārbaudīt datu avotus un pārliecināties, ka tie ir pareizi, pirms turpinām ar datu ielādi
# assert False, "Vēl nelasam datus"
sales_january_raw = pd.read_csv(january_csv_source)
sales_february_raw = pd.read_csv(february_csv_source)
products_df = pd.read_json(products_json_source)

try:
    stores_df = pd.read_html(stores_html_source)[0]
    stores_source = "HTML table"
except (ImportError, ValueError):
    print(f"Neizdevās ielādēt veikalus no HTML, pārejam uz CSV avotu: {stores_html_source}")
    stores_df = pd.read_csv(stores_csv_source)
    stores_source = "CSV fallback"

sqlite_cache_path = ensure_local_binary(sqlite_source, "regional_targets.sqlite")
# Python atbalsta jau iebūvētu atbalstu SQLite datubāzēm, tāpēc mēs varam izmantot sqlite3 moduli, lai izveidotu savienojumu ar SQLite datubāzi un izpildītu SQL vaicājumus, lai ielādētu datus tieši DataFrame formātā ar pd.read_sql_query funkciju
# ja jums ir citas SQL databāzes, piemēram, MySQL, PostgreSQL, SQL Server utt., 
# jūs varat izmantot atbilstošos Python bibliotēkas (piemēram, pymysql, psycopg2, pyodbc) un SQLAlchemy, lai izveidotu savienojumu un ielādētu datus līdzīgi kā ar SQLite, bet ar atbilstošiem savienojuma parametriem un bibliotēkas funkcijām
# tur nāks klāt lietotāja vārds un parole
# sintakse principā paliks tā pati
with sqlite3.connect(sqlite_cache_path) as connection:
    targets_df = pd.read_sql_query(
        "SELECT month, region, target_revenue FROM regional_monthly_targets ORDER BY month, region",
        connection,
    )

excel_cache_path = ensure_local_binary(excel_source, "reference_tables.xlsx")
excel_tables = pd.read_excel(excel_cache_path, sheet_name=None)

loaded_objects = pd.DataFrame(
    [
        {"dataset": "sales_january_raw", "rows": len(sales_january_raw), "columns": sales_january_raw.shape[1], "source_mode": DATA_SOURCE_MODE},
        {"dataset": "sales_february_raw", "rows": len(sales_february_raw), "columns": sales_february_raw.shape[1], "source_mode": DATA_SOURCE_MODE},
        {"dataset": "products_df", "rows": len(products_df), "columns": products_df.shape[1], "source_mode": DATA_SOURCE_MODE},
        {"dataset": f"stores_df ({stores_source})", "rows": len(stores_df), "columns": stores_df.shape[1], "source_mode": DATA_SOURCE_MODE},
        {"dataset": "targets_df", "rows": len(targets_df), "columns": targets_df.shape[1], "source_mode": DATA_SOURCE_MODE},
        {"dataset": "excel_tables", "rows": len(excel_tables), "columns": len(excel_tables), "source_mode": DATA_SOURCE_MODE},
    ]
)
# informācija kas mums ir
loaded_objects


sales_january_csv: D:\Github\RTU_Python_CSP\data\day4\raw\sales_january_raw.csv
sales_february_csv: D:\Github\RTU_Python_CSP\data\day4\raw\sales_february_raw.csv
products_json: D:\Github\RTU_Python_CSP\data\day4\raw\products_catalog.json
stores_html: D:\Github\RTU_Python_CSP\data\day4\raw\stores.html
stores_csv: D:\Github\RTU_Python_CSP\data\day4\raw\stores.csv
targets_sqlite: D:\Github\RTU_Python_CSP\data\day4\raw\regional_targets.sqlite
reference_excel: D:\Github\RTU_Python_CSP\data\day4\raw\reference_tables.xlsx


,dataset,rows,columns,source_mode
0,sales_january_raw,9,10,local
1,sales_february_raw,9,10,local
2,products_df,5,5,local
3,stores_df (HTML table),5,6,local
4,targets_df,10,3,local
5,excel_tables,2,2,local


In [5]:
# drusku vairāk par ielādi no HTML tabulas, jo tas var būt nedaudz trausls process, atkarībā no HTML struktūras un izmantotajām bibliotēkām. Ja ielāde no HTML neizdodas, mēs pārejam uz CSV avotu kā rezerves variantu.url = "https://en.wikipedia.org/wiki/List_of_cities_and_towns_in_Latvia"
# url = "https://en.wikipedia.org/wiki/List_of_cities_and_towns_in_Latvia"
url = "https://data.gov.lv/dati/lv/group/valsts-parvalde"
# mēs nolasim VISAS tabulas kuras ir šajā lapā
# un tad apskatīsim cik tādas tabulas mums ir
wiki_tables = pd.read_html(url) # tiek nolašits, noparsēts un pārvērsts par datu rāmjiem (DataFrame) sarakstu
print(f"Tabulas atrastas: {len(wiki_tables)}")

Tabulas atrastas: 1


In [6]:
# tātad ja mums ir vismaz viena tad varam to apskatīt
if wiki_tables:
    display(wiki_tables[0]) # display ir kā print tieši pandas dataframe gadījumā, tas izdrukās tabulas formātā, kas ir daudz pārskatāmāk nekā parasts print

,Datu kopa,Iestāde,Pievienots,Atjaunots,Atjaunošanas biežums
0,Oficiālā elektroniskā adrese (e-adrese),Valsts digitālās attīstības aģentūra,2019-10-01,2026-04-20,pastāvīgi
1,Tiesību aktu veidi,Valsts kanceleja,2021-10-12,2026-04-20,katru dienu
2,Statistika par Latvija.gov.lv e-pakalpojumiem,Valsts digitālās attīstības aģentūra,2020-04-09,2026-04-20,citi
3,TVP tīmekļvietņu saturs,Valsts kanceleja,2020-05-08,2026-04-20,katru dienu
4,Eiropas Savienības fondu 14-20 perioda līdzfin...,Centrālā finanšu un līgumu aģentūra,2018-02-07,2026-04-19,katru dienu
5,Vakances,Nodarbinātības Valsts Aģentūra,2020-03-23,2026-04-19,katru dienu
6,Publisko personu un iestāžu saraksts,LR Uzņēmumu reģistrs,2018-06-12,2026-04-19,katru dienu
7,Iepirkumu rezultāti (e-konkursi),Valsts digitālās attīstības aģentūra,2019-07-19,2026-04-19,katru dienu
8,Iepirkumu grozījumi (e-konkursi),Valsts digitālās attīstības aģentūra,2019-07-19,2026-04-19,katru dienu
9,Izsludinātie iepirkumi (e-konkursi),Valsts digitālās attīstības aģentūra,2019-04-12,2026-04-19,katru dienu


In [7]:
# tātad ja zinam ka mums ir 5 lapas ar tabulām, tad varam tās visas nolasīt
# formāts ir url un tad page=1, page=2 utt. atkarībā no lapas numura
all_urls = []
for page_num in range(1, 6):
    page_url = f"{url}?page={page_num}"
    all_urls.append(page_url)
# izdrukājam visas adreses kuras nolasīsim
for page_url in all_urls:
    print(page_url)

https://data.gov.lv/dati/lv/group/valsts-parvalde?page=1
https://data.gov.lv/dati/lv/group/valsts-parvalde?page=2
https://data.gov.lv/dati/lv/group/valsts-parvalde?page=3
https://data.gov.lv/dati/lv/group/valsts-parvalde?page=4
https://data.gov.lv/dati/lv/group/valsts-parvalde?page=5


In [8]:
# tātad mums ir adreses (un tās ir publiski pieejamas)
# iesim cauri pa vienai un ja lapā ir tabula tad to nolasīsim un pievienosim musu sarakstam
data_gov_tables = [] # sākam ar tukšu sarakstu kurā glabāsim visas tabulas kuras nolasīsim
# mums vajadzēs mazu pauzi tāpec izmantosim Python biblioteku time un tās funkciju sleep, lai ieviestu nelielu pauzi starp pieprasījumiem, lai neuztrauktu serveri un izvairītos no iespējamām bloķēšanām
import time # nākošreiz liekam to pašā pirmā šunā
for page_url in all_urls: # ciklojam cauri visām adresēm kuras izveidojām
    try: # try mums ļauj meģinat kodu kurš pieļauj kļūdas un ja kļūda notiek, tad mēs varam to noķert un apstrādāt, nevis lai visa programma apstātos
        tables_on_page = pd.read_html(page_url) # te mums būs saraksts ar DataFrame tabulām
        data_gov_tables.extend(tables_on_page) # pievienojam VISAS tabulas no lapas
        # alternatīva būtu ņemt tikai pirmo tabulu no katras lapas, tad mēs varētu rakstīt data_gov_tables.append(tables_on_page[0]) bet šeit mēs ņemam visas
        print(f"Tabulas nolasītas no: {page_url} (atrastas {len(tables_on_page)})")
    except Exception as e:
        print(f"Kļūda nolasot tabulas no: {page_url} - {e}")
    time.sleep(1) # pauze 1 sekunde starp pieprasījumiem

# cik kopā tabulas nolasījām?
print(f"Kopā nolasītas tabulas: {len(data_gov_tables)}")

Tabulas nolasītas no: https://data.gov.lv/dati/lv/group/valsts-parvalde?page=1 (atrastas 1)
Tabulas nolasītas no: https://data.gov.lv/dati/lv/group/valsts-parvalde?page=2 (atrastas 1)
Tabulas nolasītas no: https://data.gov.lv/dati/lv/group/valsts-parvalde?page=3 (atrastas 1)
Tabulas nolasītas no: https://data.gov.lv/dati/lv/group/valsts-parvalde?page=4 (atrastas 1)
Tabulas nolasītas no: https://data.gov.lv/dati/lv/group/valsts-parvalde?page=5 (atrastas 1)
Kopā nolasītas tabulas: 5


In [9]:
# apskatam pēdejo tabulu kuru nolasījām
if data_gov_tables:
    print("Pēdējā nolasītā tabula:")
    display(data_gov_tables[-1])
else:
    print("Neizdevās nolasīt nevienu tabulu no datu.gov.lv")

Pēdējā nolasītā tabula:


,Datu kopa,Iestāde,Pievienots,Atjaunots,Atjaunošanas biežums
0,Eiropas Savienības struktūrfondu un Kohēzijas ...,Finanšu ministrija,2018-07-05,2019-10-07,katru dienu
1,Eiropas Savienības struktūrfondu un Kohēzijas ...,Finanšu ministrija,2018-07-05,2019-10-07,katru dienu
2,Domes sēžu darba kārtības,Olaines novada pašvaldība,2019-04-30,2019-05-03,reizi mēnesī
3,13.Saeimas vēlēšanas,Centrālā vēlēšanu komisija,2019-02-19,2019-02-19,nekad
4,2011. gada tautas nobalsošanas par 10.Saeimas ...,Centrālā vēlēšanu komisija,2019-02-04,2019-02-04,nekad
5,2012. gada tautas nobalsošanas par likumprojek...,Centrālā vēlēšanu komisija,2019-02-04,2019-02-04,nekad
6,2017. gada republikas pilsētas domes un novada...,Centrālā vēlēšanu komisija,2018-06-26,2018-06-26,nekad
7,2014. gada 12.Saeimas vēlēšanu rezultāti un vē...,Centrālā vēlēšanu komisija,2018-06-26,2018-06-26,nekad


## Tabulu ar identiskām kolonnām apvienošana ar `pd.concat()`
- Ja mums ir vairākas tabulas ar identiskām kolonnām, mēs varam tās apvienot vienā lielā tabulā, izmantojot `pd.concat()`. Šī funkcija ļauj mums vertikāli sapludināt vairākas DataFrame, saglabājot visas kolonnas un rindiņas. Tas ir īpaši noderīgi, ja mēs esam nolasījuši vairākas lapas ar tabulām un vēlamies tās apvienot vienā datu kopā.

In [12]:
# vispirms neliela pārbaude vai mums visām tabulām ir tās pašas kolonnas, jo tikai tad mēs varam tās apvienot vienā lielā tabulā
column_sets = [set(df.columns) for df in data_gov_tables]
unique_column_sets = set(tuple(cols) for cols in column_sets)
if len(unique_column_sets) == 1:
    print("Visām tabulām ir identiskas kolonnas, varam tās apvienot.")
# bez sets es varētu iet cauri visam tabulām no 2 un pārbaudīt vai sakrīt ar pirmo
first_dataframe_columns = data_gov_tables[0].columns
print(f"Pirmās tabulas kolonnas: {first_dataframe_columns}")
for idx, df in enumerate(data_gov_tables[1:], start=2):
    if set(df.columns) != set(first_dataframe_columns): # seciba nebūs svarīga, jo mēs tikai gribam pārbaudīt vai ir vienādas kolonnas, nevis vai ir vienādā secībā
        print(f"Tabula {idx} kolonnas atšķiras no pirmās tabulas kolonnām.")
    else:
        print(f"Tabula {idx} kolonnas sakrīt ar pirmās tabulas kolonnām.")

Visām tabulām ir identiskas kolonnas, varam tās apvienot.
Pirmās tabulas kolonnas: Index(['Datu kopa', 'Iestāde', 'Pievienots', 'Atjaunots', 'Atjaunošanas biežums'], dtype='str')
Tabula 2 kolonnas sakrīt ar pirmās tabulas kolonnām.
Tabula 3 kolonnas sakrīt ar pirmās tabulas kolonnām.
Tabula 4 kolonnas sakrīt ar pirmās tabulas kolonnām.
Tabula 5 kolonnas sakrīt ar pirmās tabulas kolonnām.


In [13]:
# ja mums jau visas tabulas ir sarakstā tad vienkārši padodam šo sarakstu pd.concat funkcijai, lai tās apvienotu vienā lielā tabulā
data_gov_df = pd.concat(data_gov_tables, ignore_index=True) # ignore_index=True mums nodrošina ka jaunajā apvienotajā tabulā rindiņu indeksi būs unikāli un secīgi, nevis katrai sākotnējai tabulai sāksies no 0
print(f"Apvienotā tabula izveidota ar {len(data_gov_df)} rindiņām un {data_gov_df.shape[1]} kolonnām.")
# apskatamies pirmās 5 rindiņas no apvienotās tabulas
# display pēdejā rindā koda šunai nav vajadzīgs
# display(data_gov_df.head()) # šo izmantojam ja vajag kādu tabulu rādīt koda šunas vidū
data_gov_df.head()

Apvienotā tabula izveidota ar 88 rindiņām un 5 kolonnām.


,Datu kopa,Iestāde,Pievienots,Atjaunots,Atjaunošanas biežums
0,Oficiālā elektroniskā adrese (e-adrese),Valsts digitālās attīstības aģentūra,2019-10-01,2026-04-20,pastāvīgi
1,Tiesību aktu veidi,Valsts kanceleja,2021-10-12,2026-04-20,katru dienu
2,Statistika par Latvija.gov.lv e-pakalpojumiem,Valsts digitālās attīstības aģentūra,2020-04-09,2026-04-20,citi
3,TVP tīmekļvietņu saturs,Valsts kanceleja,2020-05-08,2026-04-20,katru dienu
4,Eiropas Savienības fondu 14-20 perioda līdzfin...,Centrālā finanšu un līgumu aģentūra,2018-02-07,2026-04-19,katru dienu


In [14]:
# pēdejās 4 rindiņas
data_gov_df.tail(4)

,Datu kopa,Iestāde,Pievienots,Atjaunots,Atjaunošanas biežums
84,2011. gada tautas nobalsošanas par 10.Saeimas ...,Centrālā vēlēšanu komisija,2019-02-04,2019-02-04,nekad
85,2012. gada tautas nobalsošanas par likumprojek...,Centrālā vēlēšanu komisija,2019-02-04,2019-02-04,nekad
86,2017. gada republikas pilsētas domes un novada...,Centrālā vēlēšanu komisija,2018-06-26,2018-06-26,nekad
87,2014. gada 12.Saeimas vēlēšanu rezultāti un vē...,Centrālā vēlēšanu komisija,2018-06-26,2018-06-26,nekad


In [16]:
# varam uzreiz saglabāt kā csv teiksim (vai excel )
# tātad pašai DataFrame izmantojot to_csv funkciju, mēs varam saglabāt šo tabulu kā csv failu, norādot ceļu un faila nosaukumu, un index=False mums nodrošina ka saglabātajā csv failā nebūs rindiņu indeksi kā atsevišķa kolonna
data_gov_df.to_csv(OUTPUT_DIR / "apvienotie_dati.csv", index=False)

In [23]:
# paskatīsimies kas notiksies ja es pirmaja tabulai pamainišu kolonu secību
# patlaba kolonas ir
print(f"Patlaba kolonu secība: {data_gov_tables[0].columns.tolist()}")
# sakārtosim kolonas alphabetiskā secībā
first_table_columns_sorted = sorted(data_gov_tables[0].columns)
print(f"Sakārtotās kolonas: {first_table_columns_sorted}")
# šīs sakartotas kolonas es vēl nekur neizmantoju
# tagad samainīsu secību pirmajai tabulai, lai tā būtu atšķirīga no pārējām
data_gov_tables[0] = data_gov_tables[0][first_table_columns_sorted]
# jaunā secība pirmajai tabulai
print(f"Jaunā secība pirmajai tabulai: {data_gov_tables[0].columns.tolist()}")
# head
data_gov_tables[0].head()

Patlaba kolonu secība: ['Atjaunots', 'Atjaunošanas biežums', 'Datu kopa', 'Iestāde', 'Pievienots']
Sakārtotās kolonas: ['Atjaunots', 'Atjaunošanas biežums', 'Datu kopa', 'Iestāde', 'Pievienots']
Jaunā secība pirmajai tabulai: ['Atjaunots', 'Atjaunošanas biežums', 'Datu kopa', 'Iestāde', 'Pievienots']


,Atjaunots,Atjaunošanas biežums,Datu kopa,Iestāde,Pievienots
0,2026-04-20,pastāvīgi,Oficiālā elektroniskā adrese (e-adrese),Valsts digitālās attīstības aģentūra,2019-10-01
1,2026-04-20,katru dienu,Tiesību aktu veidi,Valsts kanceleja,2021-10-12
2,2026-04-20,citi,Statistika par Latvija.gov.lv e-pakalpojumiem,Valsts digitālās attīstības aģentūra,2020-04-09
3,2026-04-20,katru dienu,TVP tīmekļvietņu saturs,Valsts kanceleja,2020-05-08
4,2026-04-19,katru dienu,Eiropas Savienības fondu 14-20 perioda līdzfin...,Centrālā finanšu un līgumu aģentūra,2018-02-07


In [22]:
# pēdejai tabulai joprojam ir vecā secība
data_gov_tables[-1].head()

,Datu kopa,Iestāde,Pievienots,Atjaunots,Atjaunošanas biežums
0,Eiropas Savienības struktūrfondu un Kohēzijas ...,Finanšu ministrija,2018-07-05,2019-10-07,katru dienu
1,Eiropas Savienības struktūrfondu un Kohēzijas ...,Finanšu ministrija,2018-07-05,2019-10-07,katru dienu
2,Domes sēžu darba kārtības,Olaines novada pašvaldība,2019-04-30,2019-05-03,reizi mēnesī
3,13.Saeimas vēlēšanas,Centrālā vēlēšanu komisija,2019-02-19,2019-02-19,nekad
4,2011. gada tautas nobalsošanas par 10.Saeimas ...,Centrālā vēlēšanu komisija,2019-02-04,2019-02-04,nekad


In [21]:
# tagad apvienosim visas tabulas atkal ar pd.concat
# sagaidāms ka jaunajā tabulā kolonas izmantos pirmas tabulas secību, jo pd.concat izmanto pirmo tabulu kā "šablonu" kolonu secībai, un pārējās tabulas tiek pielāgotas šai secībai, pat ja to sākotnējā secība ir atšķirīga
data_gov_df_also = pd.concat(data_gov_tables, ignore_index=True)
print(f"Apvienotā tabula izveidota ar {len(data_gov_df_also)} rindiņām un {data_gov_df_also.shape[1]} kolonnām.")
# apskatamies pirmās 5 rindiņas no apvienotās tabulas
# display pēdejā rindā koda šunai nav vajadzīgs
data_gov_df_also.head()

Apvienotā tabula izveidota ar 88 rindiņām un 5 kolonnām.


,Atjaunots,Atjaunošanas biežums,Datu kopa,Iestāde,Pievienots
0,2026-04-20,pastāvīgi,Oficiālā elektroniskā adrese (e-adrese),Valsts digitālās attīstības aģentūra,2019-10-01
1,2026-04-20,katru dienu,Tiesību aktu veidi,Valsts kanceleja,2021-10-12
2,2026-04-20,citi,Statistika par Latvija.gov.lv e-pakalpojumiem,Valsts digitālās attīstības aģentūra,2020-04-09
3,2026-04-20,katru dienu,TVP tīmekļvietņu saturs,Valsts kanceleja,2020-05-08
4,2026-04-19,katru dienu,Eiropas Savienības fondu 14-20 perioda līdzfin...,Centrālā finanšu un līgumu aģentūra,2018-02-07


In [26]:
# noņemis pēdejai tabulai kolonu Atjaunots
# ar drop mēs noņem kolonu
# pēc noklusejam oriģināla tabula netiek modificēta
# ja gribat modificēt tad ir divi varianti
# viens ir izmantot inplace=True, kas uzreiz modificē tabulu
# otrs ir redzams nakošā rindā kur mēs tā teikt parrakstam pāri 
# vispirms pārbaudam vai šāda kolona vispār ir
if "Atjaunots" in data_gov_tables[-1].columns:
    print("Kolonna 'Atjaunots' ir pēdējā tabulā, to noņemsim.")
    data_gov_tables[-1] = data_gov_tables[-1].drop(columns=["Atjaunots"])
else:
    print("Kolonna 'Atjaunots' nav pēdējā tabulā, nav ko noņemt.")
# pēdeja tabula atkal
data_gov_tables[-1].head()

Kolonna 'Atjaunots' nav pēdējā tabulā, nav ko noņemt.


,Datu kopa,Iestāde,Pievienots,Atjaunošanas biežums
0,Eiropas Savienības struktūrfondu un Kohēzijas ...,Finanšu ministrija,2018-07-05,katru dienu
1,Eiropas Savienības struktūrfondu un Kohēzijas ...,Finanšu ministrija,2018-07-05,katru dienu
2,Domes sēžu darba kārtības,Olaines novada pašvaldība,2019-04-30,reizi mēnesī
3,13.Saeimas vēlēšanas,Centrālā vēlēšanu komisija,2019-02-19,nekad
4,2011. gada tautas nobalsošanas par 10.Saeimas ...,Centrālā vēlēšanu komisija,2019-02-04,nekad


In [27]:
# tagad skatīsimies kā izskata liela tabula apvienojot, ja nav kāda kolona kādai tabulai
data_gov_df_missing_col = pd.concat(data_gov_tables, ignore_index=True)
print(f"Apvienotā tabula izveidota ar {len(data_gov_df_missing_col)} rindiņām un {data_gov_df_missing_col.shape[1]} kolonnām.")
data_gov_df_missing_col.head()

Apvienotā tabula izveidota ar 88 rindiņām un 5 kolonnām.


,Atjaunots,Atjaunošanas biežums,Datu kopa,Iestāde,Pievienots
0,2026-04-20,pastāvīgi,Oficiālā elektroniskā adrese (e-adrese),Valsts digitālās attīstības aģentūra,2019-10-01
1,2026-04-20,katru dienu,Tiesību aktu veidi,Valsts kanceleja,2021-10-12
2,2026-04-20,citi,Statistika par Latvija.gov.lv e-pakalpojumiem,Valsts digitālās attīstības aģentūra,2020-04-09
3,2026-04-20,katru dienu,TVP tīmekļvietņu saturs,Valsts kanceleja,2020-05-08
4,2026-04-19,katru dienu,Eiropas Savienības fondu 14-20 perioda līdzfin...,Centrālā finanšu un līgumu aģentūra,2018-02-07


In [28]:
# sagaidam ka tail būs none Atjaunots kolonna
data_gov_df_missing_col.tail()

,Atjaunots,Atjaunošanas biežums,Datu kopa,Iestāde,Pievienots
83,NaN,nekad,13.Saeimas vēlēšanas,Centrālā vēlēšanu komisija,2019-02-19
84,NaN,nekad,2011. gada tautas nobalsošanas par 10.Saeimas ...,Centrālā vēlēšanu komisija,2019-02-04
85,NaN,nekad,2012. gada tautas nobalsošanas par likumprojek...,Centrālā vēlēšanu komisija,2019-02-04
86,NaN,nekad,2017. gada republikas pilsētas domes un novada...,Centrālā vēlēšanu komisija,2018-06-26
87,NaN,nekad,2014. gada 12.Saeimas vēlēšanu rezultāti un vē...,Centrālā vēlēšanu komisija,2018-06-26


In [ ]:
# es varēu arī padot tikai divas tabulas pirmo un pēdejo
# es izveidoju jaunu sarakstu ar tikai pirmo un pēdejo tabulu un padodu to pd.concat funkcijai, lai apvienotu tikai šīs divas tabulas, un atkal ignore_index=True nodrošina unikālus un secīgus rindiņu indeksus jaunajā tabulā
data_gov_df_first_last = pd.concat([data_gov_tables[0], data_gov_tables[-1]], ignore_index=True)
print(f"Apvienotā tabula (pirmā + pēdējā) izveidota ar {len(data_gov_df_first_last)} rindiņām un {data_gov_df_first_last.shape[1]} kolonnām.")
data_gov_df_first_last.head()

## 2. Inspect

Goal: understand what was loaded before changing anything.

Inspection is the stage where you learn the shape, grain, and reliability of the data. It should happen immediately after loading and before heavy cleaning. If you skip inspection, you can easily spend time cleaning the wrong columns, filtering on the wrong assumptions, or merging on unstable keys.

Fast first-pass checks:
- Preview rows with `.head()`, `.tail()`, and `.sample()`.
- Check dimensions with `.shape`.
- Review schema with `.columns`, `.dtypes`, and `.info()`.
- Generate numeric and categorical summaries with `.describe()` and `.value_counts()`.
- Measure missingness with `.isna().sum()`.
- Check uniqueness and possible keys with `.nunique()` or duplicate tests.
- Review memory footprint with `.memory_usage(deep=True)` when size matters.

Questions inspection should answer:
- What is the grain of the table: transaction, person, product, date, event, or something else?
- Which columns look like identifiers or future join keys?
- Which columns are unexpectedly `object` or `string` and may need conversion?
- Which fields contain obvious missing values, placeholders, or inconsistent categories?
- Are there duplicate rows or duplicate keys?
- Do any numeric columns show impossible values, suspicious zeros, or very large outliers?
- Which fields are good candidates for grouping, filtering, or plotting later?

Useful teaching angle:
- Treat inspection as a form of hypothesis building.
- Write down what you think each important column means.
- Mark anything uncertain so later cleaning rules are explicit rather than accidental.

Common pitfalls:
- Relying only on `.head()` and assuming the whole dataset behaves the same way.
- Missing problems hidden in the tail, random samples, or rare categories.
- Treating `object` dtype as harmless when it may hide mixed strings, numbers, and null markers.
- Confusing row count with unique entity count.

Documentation references:
- [pandas essential basic functionality](https://pandas.pydata.org/docs/user_guide/basics.html)
- [pandas.DataFrame.info](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html)
- [pandas.DataFrame.describe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)


In [ ]:
# šī recepte strādās salīdzinot jebkura veida divas tabulas
# te nav nekādu konkrētu kolonu nosaukumi, vai izmēru prasību, vienkārši salīdzinām divas tabulas un skatāmies kādas ir atšķirības datu kvalitātē, piemēram, trūkstošās vērtības, dublikāti, datu tipi utt.
display(sales_january_raw.head())
display(sales_february_raw.head())

# tiek uzbūvēta jauna tabula ar informāciju par datu kvalitāti katram ielādētajam datasetam, kurā ir kolonnas dataset, rows, missing_values un duplicate_rows, un katra rinda atbilst vienam no ielādētajiem datu rāmjiem (sales_january_raw, sales_february_raw, products_df)
inspection_summary = pd.DataFrame(
    [
        {
            "dataset": "sales_january_raw",
            "rows": len(sales_january_raw),
            "missing_values": int(sales_january_raw.isna().sum().sum()), # ar divām .sum() mēs saskaitām visas šunas
            "duplicate_rows": int(sales_january_raw.duplicated().sum()),
        },
        {
            "dataset": "sales_february_raw",
            "rows": len(sales_february_raw),
            "missing_values": int(sales_february_raw.isna().sum().sum()),
            "duplicate_rows": int(sales_february_raw.duplicated().sum()),
        },
        {
            "dataset": "products_df",
            "rows": len(products_df),
            "missing_values": int(products_df.isna().sum().sum()),
            "duplicate_rows": int(products_df.duplicated().sum()),
        },
    ]
)

january_missing = sales_january_raw.isna().sum().sort_values(ascending=False).rename("missing_count")
january_dtypes = sales_january_raw.dtypes.astype(str).rename("dtype")

display(inspection_summary)
display(january_dtypes.to_frame())
january_missing.to_frame()


,Order ID,Order Date,Store Code,Product Code,Customer Segment,Units,Unit Price,Discount %,Sales Channel,Promo Flag
0,1001,2026-01-03,S001,P100,Consumer,4,12.5,0.00,Online,yes
1,1002,2026-01-05,S002,P200,Corporate,2,85.0,0.10,Retail,no
2,1003,2026-01-07,S003,P400,Consumer,1,210.0,NaN,Online,yes
3,1004,2026/01/10,S004,P300,Home Office,3,42.0,0.05,Retail,No
4,1005,15-01-2026,S001,P100,consumer,5,12.5,0.15,Online,YES


,Order ID,Order Date,Store Code,Product Code,Customer Segment,Units,Unit Price,Discount %,Sales Channel,Promo Flag
0,2001,2026-02-02,S001,P200,Corporate,1.0,85.0,0.0,Online,no
1,2002,2026-02-05,S003,p300,Consumer,4.0,42.0,0.1,Retail,yes
2,2003,2026-02-09,S004,P400,Home Office,1.0,210.0,0.15,Retail,no
3,2004,2026/02/11,S002,P100,Consumer,6.0,12.5,0.05,Online,yes
4,2005,18-02-2026,S005,P500,Corporate,NaN,15.0,0.0,Wholesale,no


,dataset,rows,missing_values,duplicate_rows
0,sales_january_raw,9,2,1
1,sales_february_raw,9,1,0
2,products_df,5,0,0


,dtype
Order ID,int64
Order Date,str
Store Code,str
Product Code,str
Customer Segment,str
Units,int64
Unit Price,float64
Discount %,float64
Sales Channel,str
Promo Flag,str


,missing_count
Discount %,1
Customer Segment,1
Order Date,0
Order ID,0
Product Code,0
Store Code,0
Units,0
Unit Price,0
Sales Channel,0
Promo Flag,0


In [30]:
# apskatam products_df
products_df.head()

,product_code,product_name,category,subcategory,base_price
0,P100,Notebook Set,Office,Paper,12.5
1,P200,Wireless Keyboard,Electronics,Accessories,85.0
2,P300,Desk Lamp,Office,Lighting,42.0
3,P400,Monitor 24,Electronics,Display,210.0
4,P500,Coffee Beans Pack,Breakroom,Supplies,15.0


In [ ]:
# tātad būs japatīra un tikai tad varēsim apvienot (JOIN) pēc product_code

## 3. Clean

Goal: make the data analysis-ready while preserving meaning and traceability.

Cleaning is not about making the table look pretty. It is about making rules explicit. A clean table has clear column names, reliable data types, consistent missing-value handling, standardized categories, and transparent business logic. The best cleaning steps are reproducible, easy to review, and tied to an observed issue from inspection.

Common cleaning operations:
- Standardize column names with `str.strip()`, `str.lower()`, and `str.replace()`.
- Rename columns to business-friendly names with `.rename()`.
- Convert types with `.astype()`, `.convert_dtypes()`, `pd.to_numeric()`, and `pd.to_datetime()`.
- Handle missing values with `.isna()`, `.fillna()`, `.dropna()`, `.where()`, or `.combine_first()`.
- Clean text with `.str.strip()`, `.str.lower()`, `.str.upper()`, `.str.replace()`, `.str.extract()`, and `.str.contains()`.
- Remove or diagnose duplicates with `.duplicated()` and `.drop_duplicates()`.
- Create derived columns with `.assign()` or direct column expressions.
- Standardize categories, units, date formats, and boolean flags.

Cleaning decisions worth discussing:
- Should invalid values raise errors, be coerced to missing, or be kept for manual review?
- Is it better to drop rows, fill values, or create a separate quality flag?
- Are some columns raw source fields that should be preserved untouched alongside cleaned versions?
- Should missing values be filled globally, per group, or not at all?
- Which rules are business rules and which are pure technical normalization?

Practical habits:
- Start from `clean_df = df.copy()` so the raw object stays available for comparison.
- Prefer vectorized transformations over manual loops.
- Keep a short note for each major cleaning rule so you can justify it later.
- Re-run inspection after major cleaning changes.

Common pitfalls:
- Hiding bad data by filling everything with one default value.
- Parsing dates without checking locale or day-month order.
- Mixing cleaned and raw categories in the same column.
- Dropping duplicates without first deciding what counts as a duplicate.

Documentation references:
- [pandas working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas working with text data](https://pandas.pydata.org/docs/user_guide/text.html)
- [pandas.DataFrame.drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)
- [pandas.to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [pandas.to_numeric](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html)


In [33]:
sales_january_raw

,Order ID,Order Date,Store Code,Product Code,Customer Segment,Units,Unit Price,Discount %,Sales Channel,Promo Flag
0,1001,2026-01-03,S001,P100,Consumer,4,12.5,0.00,Online,yes
1,1002,2026-01-05,S002,P200,Corporate,2,85.0,0.10,Retail,no
2,1003,2026-01-07,S003,P400,Consumer,1,210.0,NaN,Online,yes
3,1004,2026/01/10,S004,P300,Home Office,3,42.0,0.05,Retail,No
4,1005,15-01-2026,S001,P100,consumer,5,12.5,0.15,Online,YES
5,1006,2026-01-18,S005,P500,Corporate,2,15.0,0.00,Wholesale,no
6,1006,2026-01-18,S005,P500,Corporate,2,15.0,0.00,Wholesale,no
7,1007,2026-01-22,S002,P999,Consumer,1,30.0,0.00,Online,yes
8,1008,2026-01-28,S006,P200,NaN,2,85.0,0.20,Retail,yes


In [36]:
def clean_sales_frame(frame: pd.DataFrame, source_month: str) -> pd.DataFrame:
    cleaned = frame.copy()
    cleaned.columns = (
        cleaned.columns.str.strip() # noņem liekās atstarpes no kolonnu nosaukumiem
        .str.lower() # normalizē kolonu nosaukumus uz maziem burtiem
        .str.replace("%", "pct", regex=False) # aizstāj procentu simbolu ar "pct", lai izvairītos no problēmām ar datu tipu un nosaukumiem
        .str.replace(" ", "_", regex=False) # aizstāj atstarpes ar pasvītrojumiem, lai kolonu nosaukumi būtu vienoti un vieglāk lietojami Python kodā
    )

    text_columns = ["store_code", "product_code", "customer_segment", "sales_channel", "promo_flag"]
    for column in text_columns:
        cleaned[column] = cleaned[column].astype("string").str.strip()

    cleaned["store_code"] = cleaned["store_code"].str.upper()
    cleaned["product_code"] = cleaned["product_code"].str.upper()
    cleaned["customer_segment"] = cleaned["customer_segment"].replace({"": pd.NA}).str.title().fillna("Unknown")
    cleaned["sales_channel"] = cleaned["sales_channel"].str.title().fillna("Unknown")
    # te mēs promo_flag kolonā nomainam tekstu yes un no uz True un False, lai būtu vieglāk strādāt ar šo informāciju kā ar loģiskām vērtībām, nevis tekstu
    cleaned["promo_flag"] = cleaned["promo_flag"].str.lower().map({"yes": True, "no": False})

    # parsed_dates = pd.to_datetime(cleaned["order_date"], errors="coerce", dayfirst=True)
    parsed_dates = pd.to_datetime(cleaned["order_date"], errors="coerce") # dayfirst=False ir noklusēta
    cleaned["order_date_invalid_flag"] = parsed_dates.isna()
    # saglabāsim arī oriģinālo datumu stringu
    cleaned["order_date_original"] = cleaned["order_date"]
    cleaned["order_date"] = parsed_dates # pārrakstu pāri oriģinālai kolonnai
    # saglabāsim arī oriģinālo datumu stringu, lai varētu analizēt kādi formāti bija un kāpēc daži datumi nevarēja tikt parsēti

    units_numeric = pd.to_numeric(cleaned["units"], errors="coerce")
    cleaned["units_missing_flag"] = units_numeric.isna()
    cleaned["units"] = units_numeric.fillna(1).astype("Int64")

    cleaned["unit_price"] = pd.to_numeric(cleaned["unit_price"], errors="coerce")

    discounts = pd.to_numeric(cleaned["discount_pct"], errors="coerce")
    cleaned["discount_missing_flag"] = discounts.isna()
    cleaned["discount_pct"] = discounts.fillna(0)

    cleaned["source_month"] = source_month
    cleaned["gross_revenue"] = cleaned["units"].astype("float64") * cleaned["unit_price"]
    cleaned["net_revenue"] = (cleaned["gross_revenue"] * (1 - cleaned["discount_pct"])).round(2)
    cleaned["order_month"] = cleaned["order_date"].dt.strftime("%Y-%m")

    cleaned = cleaned.drop_duplicates(
        subset=["order_id", "order_date", "store_code", "product_code", "sales_channel"]
    ).reset_index(drop=True)
    return cleaned

# tātad ja mums ir clean_sales_frame funkcija, tad mēs varam to izmantot lai attīrītu abus sales datu rāmjus, un pēc tam mēs varam izveidot jaunu datu rāmi ar informāciju par to cik rindu bija sākotnējā "raw" versijā un cik rindu ir pēc tīrīšanas, un cik dublikātu tika noņemts, lai redzētu kāda ir datu kvalitāte un cik daudz datu tika zaudēts tīrīšanas procesā
# mums jau var būt vairaki mēneši

sales_january_clean = clean_sales_frame(sales_january_raw, "2026-01")
sales_february_clean = clean_sales_frame(sales_february_raw, "2026-02")

cleaning_report = pd.DataFrame(
    [
        {
            "dataset": "sales_january_clean",
            "raw_rows": len(sales_january_raw),
            "clean_rows": len(sales_january_clean),
            "duplicates_removed": len(sales_january_raw) - len(sales_january_clean),
        },
        {
            "dataset": "sales_february_clean",
            "raw_rows": len(sales_february_raw),
            "clean_rows": len(sales_february_clean),
            "duplicates_removed": len(sales_february_raw) - len(sales_february_clean),
        },
    ]
)

display(cleaning_report)
sales_january_clean.head()


,dataset,raw_rows,clean_rows,duplicates_removed
0,sales_january_clean,9,8,1
1,sales_february_clean,9,9,0


,order_id,order_date,store_code,product_code,customer_segment,units,unit_price,discount_pct,sales_channel,promo_flag,order_date_invalid_flag,order_date_original,units_missing_flag,discount_missing_flag,source_month,gross_revenue,net_revenue,order_month
0,1001,2026-01-03,S001,P100,Consumer,4,12.5,0.00,Online,True,False,2026-01-03,False,False,2026-01,50.0,50.00,2026-01
1,1002,2026-01-05,S002,P200,Corporate,2,85.0,0.10,Retail,False,False,2026-01-05,False,False,2026-01,170.0,153.00,2026-01
2,1003,2026-01-07,S003,P400,Consumer,1,210.0,0.00,Online,True,False,2026-01-07,False,True,2026-01,210.0,210.00,2026-01
3,1004,NaT,S004,P300,Home Office,3,42.0,0.05,Retail,False,True,2026/01/10,False,False,2026-01,126.0,119.70,NaN
4,1005,NaT,S001,P100,Consumer,5,12.5,0.15,Online,True,True,15-01-2026,False,False,2026-01,62.5,53.12,NaN


In [37]:
# pārkārtosim january_sales_clean kolonnas lai kolonas kas sākas ar order_data butu uzreiz aiz order_id
# vispirms izveidosim jaunu kolonnu secību, kurā mēs vēlamies, lai kolonnas būtu, un tad pārrakstīsim datu rāmi ar šo jauno secību
desired_column_order = [
    "order_id",
    "order_date",
    "order_date_original",
    "order_date_invalid_flag",
    "order_month",
    "store_code",
    "product_code",
    "customer_segment",
    "sales_channel",
    "promo_flag",
    "units",
    "units_missing_flag",
    "unit_price",
    "discount_pct",
    "discount_missing_flag",
    "gross_revenue",
    "net_revenue",
    "source_month",
]
sales_january_clean = sales_january_clean[desired_column_order]
sales_january_clean.head()

,order_id,order_date,order_date_original,order_date_invalid_flag,order_month,store_code,product_code,customer_segment,sales_channel,promo_flag,units,units_missing_flag,unit_price,discount_pct,discount_missing_flag,gross_revenue,net_revenue,source_month
0,1001,2026-01-03,2026-01-03,False,2026-01,S001,P100,Consumer,Online,True,4,False,12.5,0.00,False,50.0,50.00,2026-01
1,1002,2026-01-05,2026-01-05,False,2026-01,S002,P200,Corporate,Retail,False,2,False,85.0,0.10,False,170.0,153.00,2026-01
2,1003,2026-01-07,2026-01-07,False,2026-01,S003,P400,Consumer,Online,True,1,False,210.0,0.00,True,210.0,210.00,2026-01
3,1004,NaT,2026/01/10,True,NaN,S004,P300,Home Office,Retail,False,3,False,42.0,0.05,False,126.0,119.70,2026-01
4,1005,NaT,15-01-2026,True,NaN,S001,P100,Consumer,Online,True,5,False,12.5,0.15,False,62.5,53.12,2026-01


In [38]:
# man ir datums formātā 2026/01/10
# es gribu to parsēt uz datetime formātu, lai varētu ar to strādāt kā ar datumu, nevis tekstu
date_str = "2026/01/10"
parsed_date = pd.to_datetime(date_str, errors="coerce") # errors="coerce" nozīmē ka ja datums nevar tikt parsēts, tad tas tiks aizstāts ar NaT (Not a Time), kas ir pandas veids kā apzīmēt trūkstošus vai nederīgus datuma vērtības
print(parsed_date)

2026-01-10 00:00:00


In [40]:
# tagad izmantosim order_date_invalid_flag lai apstrādātu order_date_original vērtību 
# tikai tajās rindās kur karodziņš ir True, jo tas nozīmē ka order_date ir nederīgs un nevarēja tikt parsēts, tāpēc mēs varam apskatīt order_date_original vērtību un mēģināt saprast kāpēc datums nevarēja tikt parsēts, piemēram, vai ir kāds nepareizs formāts, vai ir kādi lieki simboli, vai ir kādi neparasti datumi utt.
invalid_dates = sales_january_clean[sales_january_clean["order_date_invalid_flag"]]
# invalid_dates ir DataFrame bet tas ir skats (view) uz sales_january_clean, kas satur tikai tās rindas kur order_date_invalid_flag ir True, un mēs varam apskatīt šīs rindas un analizēt kādi datumi ir nederīgi un kāpēc
print(f"Nederīgie datumi (order_date_invalid_flag = True): {len(invalid_dates)} rindiņas")
display(invalid_dates[["order_id", "order_date_original", "order_date"]])

Nederīgie datumi (order_date_invalid_flag = True): 2 rindiņas


,order_id,order_date_original,order_date
3,1004,2026/01/10,NaT
4,1005,15-01-2026,NaT


In [46]:
# mums ir divi formāti kuri vēl jānoparsē
# YYYY/MM/DD un DD-MM-YYYY
# padosim šos formātejumus pa vienam
# šakam ar YYYY/MM/DD formātu
def parse_dates_with_formats(date_series: pd.Series, custom_formats: tuple = ("%Y/%m/%d", "%d-%m-%Y")) -> pd.Series:
    # mēģinam vispirms parsēt ar standart pandas funkciju bez norādīšanas formāta, lai redzētu cik datumu var tikt parsēti automātiski
    parsed_dates = pd.to_datetime(date_series, errors="coerce")
    formats = custom_formats # šeit mēs definējam formātus kuriem mēs gribam mēģināt parsēt datumu, %Y ir gads ar 4 cipariem, %m ir mēnesis ar 2 cipariem, %d ir diena ar 2 cipariem, un atšķirība starp formātiem ir datuma atdalītājs (slash vai dash) un secība (gads-mēnesis-diena vs diena-mēnesis-gads)
    for fmt in formats: # ejam cauri visiem mūsu definētiem formātiem un mēģinam parsēt datumu ar katru formātu, lai redzētu vai kāds no šiem formātiem atbilst mūsu datuma stringiem un ļauj veiksmīgi parsēt datumu
        # we parse with custome formats only those dates which were not successfully parsed with default parsing, to avoid overwriting already correctly parsed dates
        mask_invalid = parsed_dates.isna() # izveidojam masku ar karodziņiem kur sanāca/ nesanāca parsēt datumu, True nozīmē ka datums ir nederīgs un nevarēja tikt parsēts, False nozīmē ka datums tika veiksmīgi parsēts
        if not mask_invalid.any(): # ja nav vairs nederīgu datumu, tad varam pārtraukt ciklu, jo nav vairs ko parsēt
            break # ātrāk pārtraucam ciklu, ja nav vairs nederīgu datumu
        parsed_dates[mask_invalid] = pd.to_datetime(date_series[mask_invalid], errors="coerce", format=fmt) # mēģinām parsēt tikai tās datuma vērtības kuras vēl nav veiksmīgi parsētas, un mēs norādām konkrētu formātu, lai palielinātu izredzes veiksmīgi parsēt datumu, ja tas atbilst šim formātam
    return parsed_dates

sales_january_clean["order_date"] = parse_dates_with_formats(sales_january_clean["order_date_original"])
# apskatamies
sales_january_clean.head()

,order_id,order_date,order_date_original,order_date_invalid_flag,order_month,store_code,product_code,customer_segment,sales_channel,promo_flag,units,units_missing_flag,unit_price,discount_pct,discount_missing_flag,gross_revenue,net_revenue,source_month
0,1001,2026-01-03,2026-01-03,False,2026-01,S001,P100,Consumer,Online,True,4,False,12.5,0.00,False,50.0,50.00,2026-01
1,1002,2026-01-05,2026-01-05,False,2026-01,S002,P200,Corporate,Retail,False,2,False,85.0,0.10,False,170.0,153.00,2026-01
2,1003,2026-01-07,2026-01-07,False,2026-01,S003,P400,Consumer,Online,True,1,False,210.0,0.00,True,210.0,210.00,2026-01
3,1004,2026-01-10,2026/01/10,True,NaN,S004,P300,Home Office,Retail,False,3,False,42.0,0.05,False,126.0,119.70,2026-01
4,1005,2026-01-15,15-01-2026,True,NaN,S001,P100,Consumer,Online,True,5,False,12.5,0.15,False,62.5,53.12,2026-01


## 4. Filter

Goal: keep only the records and fields needed for the current question.

Filtering is the stage where you turn a large general-purpose table into a focused working set. It includes row filtering, column selection, sorting, ordering, and sometimes lightweight feature engineering that makes the subset easier to inspect or explain.

Common filtering patterns:
- Select columns directly with `df[[...]]`.
- Filter rows with boolean masks.
- Use `.loc[]` for label-based selection and `.iloc[]` for position-based selection.
- Combine conditions with `&`, `|`, and `~`.
- Use `.isin()` for membership tests.
- Use `.between()` for ranges.
- Use `.str.contains()` for text-based filters.
- Use `.query()` for SQL-like readability, especially in teaching.
- Use `.sort_values()`, `.sort_index()`, `.nlargest()`, and `.nsmallest()` to organize results.

Teaching prompts:
- Which rows are in scope for the analysis question?
- Which columns are inputs, outputs, labels, measures, or join keys?
- Is the filter a one-off exploration, or should it become part of the reusable pipeline?
- Would the logic be clearer as a boolean mask or a `query()` expression?

Important details:
- With boolean masks, use parentheses around each condition.
- With `query()`, use `@variable_name` to refer to Python variables.
- Sort before previewing if order matters for the story you want to tell.
- Prefer explicit column lists when handing a subset to later steps.

Common pitfalls:
- Chained filtering that becomes hard to read or debug.
- Forgetting parentheses around boolean conditions.
- Filtering before data types are fixed, especially for dates and numbers.
- Treating sorted output as if it changed the original table when it did not.

Documentation references:
- [pandas indexing and selecting data](https://pandas.pydata.org/docs/user_guide/indexing.html)
- [pandas.DataFrame.query](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html)


In [48]:
sales_january_clean

,order_id,order_date,order_date_original,order_date_invalid_flag,order_month,store_code,product_code,customer_segment,sales_channel,promo_flag,units,units_missing_flag,unit_price,discount_pct,discount_missing_flag,gross_revenue,net_revenue,source_month
0,1001,2026-01-03,2026-01-03,False,2026-01,S001,P100,Consumer,Online,True,4,False,12.5,0.00,False,50.0,50.00,2026-01
1,1002,2026-01-05,2026-01-05,False,2026-01,S002,P200,Corporate,Retail,False,2,False,85.0,0.10,False,170.0,153.00,2026-01
2,1003,2026-01-07,2026-01-07,False,2026-01,S003,P400,Consumer,Online,True,1,False,210.0,0.00,True,210.0,210.00,2026-01
3,1004,2026-01-10,2026/01/10,True,NaN,S004,P300,Home Office,Retail,False,3,False,42.0,0.05,False,126.0,119.70,2026-01
4,1005,2026-01-15,15-01-2026,True,NaN,S001,P100,Consumer,Online,True,5,False,12.5,0.15,False,62.5,53.12,2026-01
5,1006,2026-01-18,2026-01-18,False,2026-01,S005,P500,Corporate,Wholesale,False,2,False,15.0,0.00,False,30.0,30.00,2026-01
6,1007,2026-01-22,2026-01-22,False,2026-01,S002,P999,Consumer,Online,True,1,False,30.0,0.00,False,30.0,30.00,2026-01
7,1008,2026-01-28,2026-01-28,False,2026-01,S006,P200,Unknown,Retail,True,2,False,85.0,0.20,False,170.0,136.00,2026-01


In [49]:
# tātad ja zinam ka kolonu vairs nevajadzēs tad var to mes nost ar drop
# mums vairs nevajag order_date_original,	order_date_invalid_flag kolonas
# šoreiz metīsim tās nost in_place = True , tātad tiks modificēts oriģinālais datu rāmis, nevis jāparraksta pāri
sales_january_clean.drop(columns=["order_date_original", "order_date_invalid_flag"], inplace=True)
sales_january_clean.head()


,order_id,order_date,order_month,store_code,product_code,customer_segment,sales_channel,promo_flag,units,units_missing_flag,unit_price,discount_pct,discount_missing_flag,gross_revenue,net_revenue,source_month
0,1001,2026-01-03,2026-01,S001,P100,Consumer,Online,True,4,False,12.5,0.00,False,50.0,50.00,2026-01
1,1002,2026-01-05,2026-01,S002,P200,Corporate,Retail,False,2,False,85.0,0.10,False,170.0,153.00,2026-01
2,1003,2026-01-07,2026-01,S003,P400,Consumer,Online,True,1,False,210.0,0.00,True,210.0,210.00,2026-01
3,1004,2026-01-10,NaN,S004,P300,Home Office,Retail,False,3,False,42.0,0.05,False,126.0,119.70,2026-01
4,1005,2026-01-15,NaN,S001,P100,Consumer,Online,True,5,False,12.5,0.15,False,62.5,53.12,2026-01


In [51]:
sales_january_clean[["sales_channel"]].value_counts(dropna=False)

sales_channel
Online           4
Retail           3
Wholesale        1
Name: count, dtype: int64

In [ ]:
# apskatism rinds kur units > 2
sales_january_clean[sales_january_clean["units"] > 2] # liktu head() ja sagaidītu daudz rindu

,order_id,order_date,order_month,store_code,product_code,customer_segment,sales_channel,promo_flag,units,units_missing_flag,unit_price,discount_pct,discount_missing_flag,gross_revenue,net_revenue,source_month
0,1001,2026-01-03,2026-01,S001,P100,Consumer,Online,True,4,False,12.5,0.00,False,50.0,50.00,2026-01
3,1004,2026-01-10,NaN,S004,P300,Home Office,Retail,False,3,False,42.0,0.05,False,126.0,119.70,2026-01
4,1005,2026-01-15,NaN,S001,P100,Consumer,Online,True,5,False,12.5,0.15,False,62.5,53.12,2026-01


In [53]:
# mes varētu izmantot gt > vietā
# gt ir pandas funkcija kas nozīmē greater than, un tā ļauj mums salīdzināt vērtības kolonnā ar noteiktu skaitli, un tā atgriež boolean masku, kur True nozīmē ka vērtība ir lielāka par norādīto skaitli, un False nozīmē ka vērtība nav lielāka par norādīto skaitli, un mēs varam šo masku izmantot lai filtrētu datu rāmi un iegūtu tikai tās rindas kur units ir lielāks par 2
sales_january_clean[sales_january_clean["units"].gt(2)]

,order_id,order_date,order_month,store_code,product_code,customer_segment,sales_channel,promo_flag,units,units_missing_flag,unit_price,discount_pct,discount_missing_flag,gross_revenue,net_revenue,source_month
0,1001,2026-01-03,2026-01,S001,P100,Consumer,Online,True,4,False,12.5,0.00,False,50.0,50.00,2026-01
3,1004,2026-01-10,NaN,S004,P300,Home Office,Retail,False,3,False,42.0,0.05,False,126.0,119.70,2026-01
4,1005,2026-01-15,NaN,S001,P100,Consumer,Online,True,5,False,12.5,0.15,False,62.5,53.12,2026-01


In [ ]:
# trešais filtra paveid ir ar SQL stila to pašu paveikt varam
sales_january_clean.query("units > 2")
# vairāk dokumntācijas par SQL stila query: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.query.html

,order_id,order_date,order_month,store_code,product_code,customer_segment,sales_channel,promo_flag,units,units_missing_flag,unit_price,discount_pct,discount_missing_flag,gross_revenue,net_revenue,source_month
0,1001,2026-01-03,2026-01,S001,P100,Consumer,Online,True,4,False,12.5,0.00,False,50.0,50.00,2026-01
3,1004,2026-01-10,NaN,S004,P300,Home Office,Retail,False,3,False,42.0,0.05,False,126.0,119.70,2026-01
4,1005,2026-01-15,NaN,S001,P100,Consumer,Online,True,5,False,12.5,0.15,False,62.5,53.12,2026-01


In [ ]:
analysis_columns = [
    "order_id",
    "order_date",
    "order_month",
    "store_code",
    "product_code",
    "customer_segment",
    "units",
    "unit_price",
    "discount_pct",
    "sales_channel",
    "promo_flag",
    "net_revenue",
    "source_month",
]


def filter_sales_frame(frame: pd.DataFrame) -> pd.DataFrame:
    filtered = frame.loc[
        frame["order_date"].notna() # ir datums
        & frame["units"].gt(0) # units > 0
        & frame["unit_price"].gt(0) # unit_price > 0
        & frame["sales_channel"].isin(["Online", "Retail"]), # sales_channel ir Online vai Retail
        analysis_columns,
    ].copy()
    return filtered.sort_values(["order_date", "order_id"]).reset_index(drop=True)


sales_january_filtered = filter_sales_frame(sales_january_clean)
sales_february_filtered = filter_sales_frame(sales_february_clean)

allowed_channels = ["Online", "Retail"]
sql_style_preview = sales_february_clean.query(
    "sales_channel in @allowed_channels and unit_price > 0"
)[["order_id", "sales_channel", "unit_price", "net_revenue"]].head()

filter_report = pd.DataFrame(
    [
        {"dataset": "sales_january_filtered", "rows_after_filter": len(sales_january_filtered)},
        {"dataset": "sales_february_filtered", "rows_after_filter": len(sales_february_filtered)},
    ]
)

display(filter_report)
display(sql_style_preview)
sales_january_filtered.head()


,dataset,rows_after_filter
0,sales_january_filtered,7
1,sales_february_filtered,7


,order_id,sales_channel,unit_price,net_revenue
0,2001,Online,85.0,85.00
1,2002,Retail,42.0,151.20
2,2003,Retail,210.0,178.50
3,2004,Online,12.5,71.25
5,2006,Online,42.0,84.00


,order_id,order_date,order_month,store_code,product_code,customer_segment,units,unit_price,discount_pct,sales_channel,promo_flag,net_revenue,source_month
0,1001,2026-01-03,2026-01,S001,P100,Consumer,4,12.5,0.00,Online,True,50.00,2026-01
1,1002,2026-01-05,2026-01,S002,P200,Corporate,2,85.0,0.10,Retail,False,153.00,2026-01
2,1003,2026-01-07,2026-01,S003,P400,Consumer,1,210.0,0.00,Online,True,210.00,2026-01
3,1004,2026-01-10,NaN,S004,P300,Home Office,3,42.0,0.05,Retail,False,119.70,2026-01
4,1005,2026-01-15,NaN,S001,P100,Consumer,5,12.5,0.15,Online,True,53.12,2026-01


## 5. Combine

Goal: integrate related data sources without losing control of row meaning.

Combining is where many pandas workflows become fragile. Row counts can explode, keys can mismatch, and null-heavy joins can silently hide data quality problems. This is why cleaning and inspection need to happen before serious merging.

Main combination patterns:
- `pd.concat()` for vertical stacking of similar extracts, such as monthly files.
- `pd.concat(..., axis=1)` for side-by-side alignment by index when that is intentional.
- `pd.merge()` for relational joins between fact and lookup tables.
- `.join()` as a convenience method for index-based joins.
- `pd.merge_asof()` for nearest-key or time-aware matching.
- `pd.merge_ordered()` for ordered data, often in time-oriented workflows.

Join choices worth demonstrating:
- `inner` join when you need only matched records.
- `left` join when one table is primary and you want to preserve all its rows.
- `right` join when the secondary table should define the final coverage.
- `outer` join when reconciliation is more important than row preservation simplicity.

Validation habits:
- Compare row counts before and after the merge.
- Check for duplicated keys on both sides before joining.
- Use `validate=` when you know the expected relationship, such as `1:1` or `m:1`.
- Use `indicator=True` when you want to diagnose unmatched records.
- Review null patterns in the newly joined columns.

Common pitfalls:
- Merging on keys with inconsistent types or formatting.
- Accidentally creating many-to-many joins and multiplying rows.
- Concatenating extracts with different schemas without checking column alignment.
- Assuming an outer join is safer when it may create a much harder table to interpret.

Documentation references:
- [pandas merge, join, concatenate and compare user guide](https://pandas.pydata.org/docs/user_guide/merging.html)
- [pandas.merge](https://pandas.pydata.org/docs/reference/api/pandas.merge.html)
- [pandas.concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)


In [57]:
# vispirms mums jānormalizē product_code kolona
display(products_df)
print("Normalizēsim product_code kolonnu, lai tā būtu lielajiem burtiem un bez liekām atstarpēm, lai varētu to izmantot kā atslēgu apvienošanai (JOIN) ar sales datiem."    )
products_lookup = products_df.copy()
products_lookup["product_code"] = products_lookup["product_code"].astype("string").str.upper()
products_lookup

,product_code,product_name,category,subcategory,base_price
0,P100,Notebook Set,Office,Paper,12.5
1,P200,Wireless Keyboard,Electronics,Accessories,85.0
2,P300,Desk Lamp,Office,Lighting,42.0
3,P400,Monitor 24,Electronics,Display,210.0
4,P500,Coffee Beans Pack,Breakroom,Supplies,15.0


Normalizēsim product_code kolonnu, lai tā būtu lielajiem burtiem un bez liekām atstarpēm, lai varētu to izmantot kā atslēgu apvienošanai (JOIN) ar sales datiem.


,product_code,product_name,category,subcategory,base_price
0,P100,Notebook Set,Office,Paper,12.5
1,P200,Wireless Keyboard,Electronics,Accessories,85.0
2,P300,Desk Lamp,Office,Lighting,42.0
3,P400,Monitor 24,Electronics,Display,210.0
4,P500,Coffee Beans Pack,Breakroom,Supplies,15.0


In [58]:
sales_january_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   order_id               8 non-null      int64         
 1   order_date             8 non-null      datetime64[us]
 2   order_month            6 non-null      str           
 3   store_code             8 non-null      string        
 4   product_code           8 non-null      string        
 5   customer_segment       8 non-null      string        
 6   sales_channel          8 non-null      string        
 7   promo_flag             8 non-null      bool          
 8   units                  8 non-null      Int64         
 9   units_missing_flag     8 non-null      bool          
 10  unit_price             8 non-null      float64       
 11  discount_pct           8 non-null      float64       
 12  discount_missing_flag  8 non-null      bool          
 13  gross_revenue       

In [59]:
sales_january_clean.head()

,order_id,order_date,order_month,store_code,product_code,customer_segment,sales_channel,promo_flag,units,units_missing_flag,unit_price,discount_pct,discount_missing_flag,gross_revenue,net_revenue,source_month
0,1001,2026-01-03,2026-01,S001,P100,Consumer,Online,True,4,False,12.5,0.00,False,50.0,50.00,2026-01
1,1002,2026-01-05,2026-01,S002,P200,Corporate,Retail,False,2,False,85.0,0.10,False,170.0,153.00,2026-01
2,1003,2026-01-07,2026-01,S003,P400,Consumer,Online,True,1,False,210.0,0.00,True,210.0,210.00,2026-01
3,1004,2026-01-10,NaN,S004,P300,Home Office,Retail,False,3,False,42.0,0.05,False,126.0,119.70,2026-01
4,1005,2026-01-15,NaN,S001,P100,Consumer,Online,True,5,False,12.5,0.15,False,62.5,53.12,2026-01


In [60]:
# pirms pārveidošanas un kopēšanas
print("Pirms pārveidošanas un normalizācijas:")
display(stores_df)
stores_lookup = stores_df.copy()
stores_lookup.columns = (
    stores_lookup.columns.astype("string")
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
stores_lookup["store_code"] = stores_lookup["store_code"].astype("string").str.upper()
print("Pēc pārveidošanas un normalizācijas:")
display(stores_lookup)

Pirms pārveidošanas un normalizācijas:


,store_code,store_name,region,city,opened_year,store_type
0,S001,Riga Central,Riga,Riga,2018,Flagship
1,S002,Liepaja Coast,Kurzeme,Liepaja,2020,Retail
2,S003,Daugavpils East,Latgale,Daugavpils,2019,Retail
3,S004,Cesis North,Vidzeme,Cesis,2021,Outlet
4,S005,Jelgava South,Zemgale,Jelgava,2017,Warehouse


Pēc pārveidošanas un normalizācijas:


,store_code,store_name,region,city,opened_year,store_type
0,S001,Riga Central,Riga,Riga,2018,Flagship
1,S002,Liepaja Coast,Kurzeme,Liepaja,2020,Retail
2,S003,Daugavpils East,Latgale,Daugavpils,2019,Retail
3,S004,Cesis North,Vidzeme,Cesis,2021,Outlet
4,S005,Jelgava South,Zemgale,Jelgava,2017,Warehouse


In [61]:
# tagad izveidojam sales tabulu ar concat
sales_all = pd.concat([sales_january_filtered, sales_february_filtered], ignore_index=True)
# shape mums parāda cik rindu un kolonu ir datu rāmī, šajā gadījumā mums ir 2000 rindu un 17 kolonnu
print(f"Apvienotā sales tabula izveidota ar {sales_all.shape[0]} rindiņām un {sales_all.shape[1]} kolonnām.")
sales_all.head()

Apvienotā sales tabula izveidota ar 14 rindiņām un 13 kolonnām.


,order_id,order_date,order_month,store_code,product_code,customer_segment,units,unit_price,discount_pct,sales_channel,promo_flag,net_revenue,source_month
0,1001,2026-01-03,2026-01,S001,P100,Consumer,4,12.5,0.00,Online,True,50.00,2026-01
1,1002,2026-01-05,2026-01,S002,P200,Corporate,2,85.0,0.10,Retail,False,153.00,2026-01
2,1003,2026-01-07,2026-01,S003,P400,Consumer,1,210.0,0.00,Online,True,210.00,2026-01
3,1004,2026-01-10,NaN,S004,P300,Home Office,3,42.0,0.05,Retail,False,119.70,2026-01
4,1005,2026-01-15,NaN,S001,P100,Consumer,5,12.5,0.15,Online,True,53.12,2026-01


In [ ]:
products_lookup = products_df.copy()
products_lookup["product_code"] = products_lookup["product_code"].astype("string").str.upper()

stores_lookup = stores_df.copy()
stores_lookup.columns = (
    stores_lookup.columns.astype("string")
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
stores_lookup["store_code"] = stores_lookup["store_code"].astype("string").str.upper()

# vispirms savelkam vertikāli janvāri un februāri
# atceramies ka es pamainiju kolonu secību janvāri bet tam nevajadzētu būt problēmu, jo pd.concat izmanto pirmo tabulu kā "šablonu" kolonu secībai, un pārējās tabulas tiek pielāgotas šai secībai, pat ja to sākotnējā secība ir atšķirīga
sales_all = pd.concat([sales_january_filtered, sales_february_filtered], ignore_index=True)
# un tagad mēs apvienojam horizontāli ar produkta un veikala informāciju, lai iegūtu bagātīgāku datu rāmi, kurā katrai pārdošanas transakcijai ir pievienota arī produkta un veikala informācija, ko mēs varam izmantot tālākai analīzei, piemēram, lai analizētu pārdošanas apjomus pēc produktu kategorijām vai veikalu reģioniem
sales_with_products = sales_all.merge(
    products_lookup,
    on="product_code", # tātad šai kolonnai būtu jabūt abās savienotās tabulās
    how="left",
    validate="m:1",
    indicator="product_match",
)
sales_enriched = sales_with_products.merge(
    stores_lookup,
    on="store_code",
    how="left",
    validate="m:1",
    indicator="store_match",
)
sales_analysis = sales_enriched.query(
    "product_match == 'both' and store_match == 'both'"
).copy()
sales_analysis["month"] = sales_analysis["order_date"].dt.strftime("%Y-%m")

merge_report = pd.DataFrame(
    [
        {"check": "rows_after_concat", "value": len(sales_all)},
        {"check": "product_match_both", "value": int((sales_enriched["product_match"] == "both").sum())},
        {"check": "product_match_left_only", "value": int((sales_enriched["product_match"] == "left_only").sum())},
        {"check": "store_match_both", "value": int((sales_enriched["store_match"] == "both").sum())},
        {"check": "store_match_left_only", "value": int((sales_enriched["store_match"] == "left_only").sum())},
        {"check": "rows_for_analysis", "value": len(sales_analysis)},
    ]
)

display(merge_report)
sales_analysis.head()


In [62]:
sales_with_products = sales_all.merge(
    products_lookup,
    on="product_code", # tātad šai kolonnai būtu jabūt abās savienotās tabulās
    how="left",
    validate="m:1",
    indicator="product_match",
)
# izmērs pēc merge
print(f"Tabula pēc produkta informācijas pievienošanas: {sales_with_products.shape[0]} rindiņas, {sales_with_products.shape[1]} kolonnas.")
sales_with_products.head()

Tabula pēc produkta informācijas pievienošanas: 14 rindiņas, 18 kolonnas.


,order_id,order_date,order_month,store_code,product_code,customer_segment,units,unit_price,discount_pct,sales_channel,promo_flag,net_revenue,source_month,product_name,category,subcategory,base_price,product_match
0,1001,2026-01-03,2026-01,S001,P100,Consumer,4,12.5,0.00,Online,True,50.00,2026-01,Notebook Set,Office,Paper,12.5,both
1,1002,2026-01-05,2026-01,S002,P200,Corporate,2,85.0,0.10,Retail,False,153.00,2026-01,Wireless Keyboard,Electronics,Accessories,85.0,both
2,1003,2026-01-07,2026-01,S003,P400,Consumer,1,210.0,0.00,Online,True,210.00,2026-01,Monitor 24,Electronics,Display,210.0,both
3,1004,2026-01-10,NaN,S004,P300,Home Office,3,42.0,0.05,Retail,False,119.70,2026-01,Desk Lamp,Office,Lighting,42.0,both
4,1005,2026-01-15,NaN,S001,P100,Consumer,5,12.5,0.15,Online,True,53.12,2026-01,Notebook Set,Office,Paper,12.5,both


In [ ]:
# pieliekam klāt arī veikalu detallizētāku informāciju
sales_enriched = sales_with_products.merge(
    stores_lookup,
    on="store_code", # kolona kurai jābūt abās tabulās, lai varētu apvienot
    how="left",
    validate="m:1",
    indicator="store_match",
)
# izmērs
print(f"""Tabula pēc veikala informācijas pievienošanas: 
      {sales_enriched.shape[0]} rindiņas, 
      {sales_enriched.shape[1]} kolonnas.""")
sales_enriched.head()

Tabula pēc veikala informācijas pievienošanas: 
      14 rindiņas, 
      24 kolonnas.


,order_id,order_date,order_month,store_code,product_code,customer_segment,units,unit_price,discount_pct,sales_channel,promo_flag,net_revenue,source_month,product_name,category,subcategory,base_price,product_match,store_name,region,city,opened_year,store_type,store_match
0,1001,2026-01-03,2026-01,S001,P100,Consumer,4,12.5,0.00,Online,True,50.00,2026-01,Notebook Set,Office,Paper,12.5,both,Riga Central,Riga,Riga,2018.0,Flagship,both
1,1002,2026-01-05,2026-01,S002,P200,Corporate,2,85.0,0.10,Retail,False,153.00,2026-01,Wireless Keyboard,Electronics,Accessories,85.0,both,Liepaja Coast,Kurzeme,Liepaja,2020.0,Retail,both
2,1003,2026-01-07,2026-01,S003,P400,Consumer,1,210.0,0.00,Online,True,210.00,2026-01,Monitor 24,Electronics,Display,210.0,both,Daugavpils East,Latgale,Daugavpils,2019.0,Retail,both
3,1004,2026-01-10,NaN,S004,P300,Home Office,3,42.0,0.05,Retail,False,119.70,2026-01,Desk Lamp,Office,Lighting,42.0,both,Cesis North,Vidzeme,Cesis,2021.0,Outlet,both
4,1005,2026-01-15,NaN,S001,P100,Consumer,5,12.5,0.15,Online,True,53.12,2026-01,Notebook Set,Office,Paper,12.5,both,Riga Central,Riga,Riga,2018.0,Flagship,both


In [64]:
# tagad saglabājam šo apvienoto un bagātināto tabulu kā csv failu, lai varētu to izmantot tālākai analīzei, piemēram, lai veiktu pārdošanas analīzi pēc produktu kategorijām, veikalu reģioniem, klientu segmentiem utt., un mēs norādām index=False, lai saglabātajā csv failā nebūtu rindiņu indeksu kā atsevišķa kolonna
print(f"Mape kur glabājam izvaddatus: {OUTPUT_DIR}")
# varat nomainīt to uz ko citu
sales_enriched.to_csv(OUTPUT_DIR / "sales_enriched.csv", index=False)
# arī kā excel
sales_enriched.to_excel(OUTPUT_DIR / "sales_enriched.xlsx", index=False)

Mape kur glabājam izvaddatus: D:\Github\RTU_Python_CSP\data\day4\outputs


## 6. Summarize

Goal: turn many detailed rows into compact, decision-ready metrics.

Summarization is where raw records become insight. A good summary respects the table grain, groups by meaningful categories, uses measures that answer a question, and produces output that can be checked against business expectations.

Common summary patterns:
- `.groupby()` with `.sum()`, `.mean()`, `.median()`, `.min()`, `.max()`, `.count()`, `.size()`, or `.nunique()`.
- `.agg()` with multiple named aggregations so the output columns are readable.
- `.transform()` when you need group-level metrics back on each original row.
- `.value_counts()` for quick frequency summaries.
- `pd.crosstab()` for contingency-style tables.
- `pd.pivot_table()` for report-friendly, Excel-like summaries.
- Time-based summaries with `.resample()` when a datetime index or column is available.

Questions to ask before aggregating:
- What is the business unit of the measure: rows, customers, orders, revenue, hours, events?
- Do you need counts, sums, averages, distinct counts, or a mixture of them?
- Should missing categories be dropped or kept visible?
- Will the summary be used for reporting, charting, validation, or another merge?

Teaching opportunities:
- Compare `.count()` and `.size()` because they answer different questions.
- Show why named aggregations are easier to read than unnamed multi-level columns.
- Connect `pivot_table()` to what learners may already know from spreadsheet PivotTables.

Common pitfalls:
- Aggregating before you understand the row grain.
- Using averages where weighted logic or counts would be more honest.
- Producing summaries that cannot be traced back to the source rows.
- Forgetting to sort the final summary before presenting it.

Documentation references:
- [pandas group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [pandas.pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)
- [pandas.crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)


In [4]:
targets_clean = targets_df.copy() # izveido pilnu kopiju dataframe
# ja ir ļoti liela datu kopa, tas protams arī aizņems operativo atmiņu
# piemēram 50M rindiņas ar vairāk kolonām tas varētu aizņemt vairākus GB atmiņās
# atkarīgs protams no tā kas glabājās kolonnās, vai int64 vai string, jo string parasti aizņem vairāk atmiņas nekā skaitlis
targets_clean.head()

,month,region,target_revenue
0,2026-01,Kurzeme,160.0
1,2026-01,Latgale,200.0
2,2026-01,Riga,110.0
3,2026-01,Vidzeme,130.0
4,2026-01,Zemgale,40.0


In [10]:
# varam groupēt pa rajoniem un veikt agregāciju target_revenu
targets_by_region = targets_clean.groupby("region")["target_revenue"].sum().reset_index().sort_values("target_revenue", ascending=False)
targets_by_region.head()

,region,target_revenue
1,Latgale,510.0
0,Kurzeme,460.0
3,Vidzeme,320.0
2,Riga,300.0
4,Zemgale,75.0


In [6]:
# kāds ir tabulas apjoms - shape
print(f"Target clean izmērs {targets_clean.shape}")

Target clean izmērs (10, 3)


In [12]:
# ielādēsim iepriekš savilktos datus
# mums dati ir šeit: data\day4\outputs\sales_enriched.xlsx
# šis ceļs ir relatīvs pret projekta sakni
# bet mūsu piezīmju grāmata atrodas notebooks mapē
# atceramies ka data ir mums blakus notebook
sales_enriched_path = Path("../data/day4/outputs/sales_enriched.xlsx")
# pārbaudam vai eksistē šāds ceļs
assert sales_enriched_path.exists(), f"Fails netika atrasts: {sales_enriched_path}"
# tātad šeit ir garantija, ka fails ekistē
df = pd.read_excel(sales_enriched_path)
print(f"Sales enriched datu rāmis ielādēts ar {df.shape[0]} rindiņām un {df.shape[1]} kolonnām.")
df.head()


Sales enriched datu rāmis ielādēts ar 14 rindiņām un 24 kolonnām.


,order_id,order_date,order_month,store_code,product_code,customer_segment,units,unit_price,discount_pct,sales_channel,promo_flag,net_revenue,source_month,product_name,category,subcategory,base_price,product_match,store_name,region,city,opened_year,store_type,store_match
0,1001,2026-01-03,2026-01,S001,P100,Consumer,4,12.5,0.00,Online,True,50.00,2026-01,Notebook Set,Office,Paper,12.5,both,Riga Central,Riga,Riga,2018.0,Flagship,both
1,1002,2026-01-05,2026-01,S002,P200,Corporate,2,85.0,0.10,Retail,False,153.00,2026-01,Wireless Keyboard,Electronics,Accessories,85.0,both,Liepaja Coast,Kurzeme,Liepaja,2020.0,Retail,both
2,1003,2026-01-07,2026-01,S003,P400,Consumer,1,210.0,0.00,Online,True,210.00,2026-01,Monitor 24,Electronics,Display,210.0,both,Daugavpils East,Latgale,Daugavpils,2019.0,Retail,both
3,1004,2026-01-10,NaN,S004,P300,Home Office,3,42.0,0.05,Retail,False,119.70,2026-01,Desk Lamp,Office,Lighting,42.0,both,Cesis North,Vidzeme,Cesis,2021.0,Outlet,both
4,1005,2026-01-15,NaN,S001,P100,Consumer,5,12.5,0.15,Online,True,53.12,2026-01,Notebook Set,Office,Paper,12.5,both,Riga Central,Riga,Riga,2018.0,Flagship,both


In [17]:
# grupēsim df pēc category and subcategory kolonām 
# agregēsim units and net_revenue
sales_summary = df.groupby(["category", "subcategory"]).agg(
    total_units_sold=pd.NamedAgg(column="units", aggfunc="sum"),
    total_net_revenue=pd.NamedAgg(column="net_revenue", aggfunc="sum"),
    count_of_subcategories=pd.NamedAgg(column="subcategory", aggfunc="count"),
    average_per_subcategory=pd.NamedAgg(column="net_revenue", aggfunc="mean"),
).reset_index().sort_values("total_net_revenue", ascending=False)
sales_summary

,category,subcategory,total_units_sold,total_net_revenue,count_of_subcategories,average_per_subcategory
0,Electronics,Accessories,8,616.25,4,154.06
1,Electronics,Display,3,556.50,3,185.50
2,Office,Lighting,9,354.90,3,118.30
3,Office,Paper,11,128.12,3,42.71


In [15]:
# es varētu uzreiz veidot jaunu kolonu avg_sale_per_unit
sales_summary["avg_sale_per_unit"] = sales_summary["total_net_revenue"] / sales_summary["total_units_sold"]
sales_summary.head()

,category,subcategory,total_units_sold,total_net_revenue,avg_sale_per_unit
0,Electronics,Accessories,8,616.25,77.03
1,Electronics,Display,3,556.50,185.50
2,Office,Lighting,9,354.90,39.43
3,Office,Paper,11,128.12,11.65


In [ ]:
# standard aggregation functions in Pandas are:
# sum, count, mean, median, min, max, std, var, first, last, nunique, etc.
# very similar to what SQL offers

In [7]:
targets_clean = targets_df.copy()
targets_clean["month"] = targets_clean["month"].astype("string")
targets_clean["region"] = targets_clean["region"].astype("string")

category_summary = (
    sales_analysis.groupby(["month", "region", "category"], dropna=False)
    .agg(
        orders=("order_id", "nunique"),
        units_sold=("units", "sum"),
        revenue=("net_revenue", "sum"),
    )
    .reset_index()
    .sort_values(["month", "revenue"], ascending=[True, False])
)

region_month_summary = (
    sales_analysis.groupby(["month", "region"], dropna=False)
    .agg(
        orders=("order_id", "nunique"),
        units_sold=("units", "sum"),
        revenue=("net_revenue", "sum"),
        average_order_value=("net_revenue", "mean"),
    )
    .reset_index()
    .round(2)
)

region_target_summary = region_month_summary.merge(
    targets_clean,
    on=["month", "region"],
    how="left",
    validate="1:1",
)
region_target_summary["achievement_pct"] = (
    region_target_summary["revenue"] / region_target_summary["target_revenue"] * 100
).round(1)

revenue_pivot = pd.pivot_table(
    region_target_summary,
    index="region",
    columns="month",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
).round(2)

display(category_summary)
display(region_target_summary)
revenue_pivot


NameError: name 'sales_analysis' is not defined

## 7. Reshape

Goal: convert the table into the layout needed for reporting, plotting, or downstream analysis.

Reshaping is the bridge between a technically correct table and a convenient table. Analysts often need long format for plotting and modeling, while managers often prefer wide format for reports. Knowing when to pivot and when to melt is one of the most useful workflow skills in pandas.

Core reshape tools:
- `pivot()` when each index-column pair has a single value.
- `pivot_table()` when duplicates need aggregation.
- `pd.melt()` when turning wide columns into row labels.
- `.stack()` and `.unstack()` for index-based reshaping.
- `.explode()` when a column contains list-like values that need row expansion.

When wide format helps:
- Side-by-side comparison by month, region, category, or scenario.
- Report tables that should look like spreadsheet summaries.
- Heatmap-style matrices built from summarized data.

When long format helps:
- Seaborn-style plotting where one column stores the measure and another stores the category.
- Multi-series visualizations where color, facet, or style comes from a variable column.
- Tidy workflows where each variable has one column and each observation has one row.

Common pitfalls:
- Using `pivot()` when duplicates exist and aggregation is actually required.
- Forgetting to rename `variable` and `value` columns after `melt()`.
- Reshaping too early and making cleaning harder.
- Building a wide table that becomes harder to merge or filter later.

Documentation references:
- [pandas reshaping and pivot tables user guide](https://pandas.pydata.org/docs/user_guide/reshaping.html)
- [pandas.melt](https://pandas.pydata.org/docs/reference/api/pandas.melt.html)
- [pandas.pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)


In [18]:
df.head()

,order_id,order_date,order_month,store_code,product_code,customer_segment,units,unit_price,discount_pct,sales_channel,promo_flag,net_revenue,source_month,product_name,category,subcategory,base_price,product_match,store_name,region,city,opened_year,store_type,store_match
0,1001,2026-01-03,2026-01,S001,P100,Consumer,4,12.5,0.00,Online,True,50.00,2026-01,Notebook Set,Office,Paper,12.5,both,Riga Central,Riga,Riga,2018.0,Flagship,both
1,1002,2026-01-05,2026-01,S002,P200,Corporate,2,85.0,0.10,Retail,False,153.00,2026-01,Wireless Keyboard,Electronics,Accessories,85.0,both,Liepaja Coast,Kurzeme,Liepaja,2020.0,Retail,both
2,1003,2026-01-07,2026-01,S003,P400,Consumer,1,210.0,0.00,Online,True,210.00,2026-01,Monitor 24,Electronics,Display,210.0,both,Daugavpils East,Latgale,Daugavpils,2019.0,Retail,both
3,1004,2026-01-10,NaN,S004,P300,Home Office,3,42.0,0.05,Retail,False,119.70,2026-01,Desk Lamp,Office,Lighting,42.0,both,Cesis North,Vidzeme,Cesis,2021.0,Outlet,both
4,1005,2026-01-15,NaN,S001,P100,Consumer,5,12.5,0.15,Online,True,53.12,2026-01,Notebook Set,Office,Paper,12.5,both,Riga Central,Riga,Riga,2018.0,Flagship,both


In [19]:
targets_clean

,month,region,target_revenue
0,2026-01,Kurzeme,160.0
1,2026-01,Latgale,200.0
2,2026-01,Riga,110.0
3,2026-01,Vidzeme,130.0
4,2026-01,Zemgale,40.0
5,2026-02,Kurzeme,300.0
6,2026-02,Latgale,310.0
7,2026-02,Riga,190.0
8,2026-02,Vidzeme,190.0
9,2026-02,Zemgale,35.0


In [21]:
# let's add a duplicate of last row to the end of targets_clean
targets_with_duplicate = pd.concat([targets_clean, targets_clean.tail(1)], ignore_index=True)
print(f"Original targets_clean shape: {targets_clean.shape}")
print(f"Shape with duplicate: {targets_with_duplicate.shape}")
targets_with_duplicate

Original targets_clean shape: (10, 3)
Shape with duplicate: (11, 3)


,month,region,target_revenue
0,2026-01,Kurzeme,160.0
1,2026-01,Latgale,200.0
2,2026-01,Riga,110.0
3,2026-01,Vidzeme,130.0
4,2026-01,Zemgale,40.0
5,2026-02,Kurzeme,300.0
6,2026-02,Latgale,310.0
7,2026-02,Riga,190.0
8,2026-02,Vidzeme,190.0
9,2026-02,Zemgale,35.0


In [24]:
# now we want to adjust the last value so index 10 and column target_revenu to 150
targets_with_duplicate.at[10, "target_revenue"] = 250
# targets_with_duplicate.at[10, "target_revenue"] += 250 # this would add 250 to existing value
targets_with_duplicate

,month,region,target_revenue
0,2026-01,Kurzeme,160.0
1,2026-01,Latgale,200.0
2,2026-01,Riga,110.0
3,2026-01,Vidzeme,130.0
4,2026-01,Zemgale,40.0
5,2026-02,Kurzeme,300.0
6,2026-02,Latgale,310.0
7,2026-02,Riga,190.0
8,2026-02,Vidzeme,190.0
9,2026-02,Zemgale,35.0


In [25]:
# we will perform a pivot on target_clean
# we will pivot region column to be columns, month to be index and target_revenue to be values
targets_pivot = pd.pivot_table(
    targets_with_duplicate,
    index="month",
    columns="region",
    values="target_revenue",
    aggfunc="sum", # since we have only one value per month and region, sum will just return that value, but if there were multiple values it would sum them up
    fill_value=0, # if there is no target for a given month and region, we will fill it with 0
).round(2) # round to 2 decimal places for better readability
targets_pivot

region,Kurzeme,Latgale,Riga,Vidzeme,Zemgale
month,,,,,
2026-01,160.0,200.0,110.0,130.0,40.0
2026-02,300.0,310.0,190.0,190.0,285.0


In [20]:
# we will perform a pivot on target_clean
# we will pivot region column to be columns, month to be index and target_revenue to be values
targets_pivot = pd.pivot_table(
    targets_clean,
    index="month",
    columns="region",
    values="target_revenue",
    aggfunc="sum", # since we have only one value per month and region, sum will just return that value, but if there were multiple values it would sum them up
    fill_value=0, # if there is no target for a given month and region, we will fill it with 0
).round(2) # round to 2 decimal places for better readability
targets_pivot

region,Kurzeme,Latgale,Riga,Vidzeme,Zemgale
month,,,,,
2026-01,160.0,200.0,110.0,130.0,40.0
2026-02,300.0,310.0,190.0,190.0,35.0


In [ ]:
report_wide = region_target_summary.pivot(index="region", columns="month", values="revenue").round(2)
metrics_long = region_target_summary.melt(
    id_vars=["region", "month"],
    value_vars=["revenue", "target_revenue", "achievement_pct"],
    var_name="metric",
    value_name="value",
).sort_values(["region", "month", "metric"]).reset_index(drop=True)

display(report_wide)
metrics_long.head(12)


## 8. Visualize

Goal: communicate an answer, not just produce a chart.

Visualization should come after the table is trustworthy enough to support a claim. In a pandas workflow, charts are usually based on cleaned and summarized data rather than directly on the raw source. This makes the visual easier to explain and much easier to validate.

Useful chart families to discuss:
- Line charts for trends over time.
- Bar charts for category comparison.
- Stacked or grouped bars for composition comparisons.
- Histograms for distribution shape.
- Box plots for spread and outliers.
- Scatter plots for relationships between two numeric fields.
- Heatmaps built from pivoted summaries.
- Small multiples or faceted views when one chart becomes too crowded.

Visualization tool choices:
- `DataFrame.plot()` and `Series.plot()` for quick pandas-native plots.
- `matplotlib` when you need fine control over axes, annotations, layouts, and styling.
- `seaborn` when a tidy long-format table is available and statistical defaults are helpful.

Chart selection cheat sheet:
- Use a bar chart when the question is "which category is larger?"
- Use a line chart when the question is "how did this change over time?"
- Use a histogram when the question is "how are values distributed?"
- Use a box plot when the question is "how do spread and outliers compare across groups?"
- Use a scatter plot when the question is "do two numeric variables move together?"
- Use a heatmap when the question is "where are the high and low pockets in a matrix?"

Preparation tips before plotting:
- Decide whether the chart should use raw rows, grouped summaries, or reshaped data.
- Sort the categories before plotting if order helps the message.
- Keep only the columns needed for the chart to reduce accidental confusion.
- Round or format values for labels after the calculation stage, not before it.
- Check whether missing values or filtered-out rows change the story.

Readability and design tips:
- Write titles that answer a business question rather than repeating the axis names.
- Use axis labels and units consistently.
- Keep legends short and place them where they do not block the data.
- Rotate labels only when necessary; if too many labels need rotation, reconsider the chart design.
- Use one highlight color intentionally and keep supporting colors quieter.
- Prefer direct value labels for short bar charts when exact numbers matter.
- Start bar charts from zero unless you have a very specific analytical reason not to.
- Use grids lightly; they should support reading, not dominate the figure.

Notebook workflow tips:
- Save important figures to files so the notebook produces reusable artifacts.
- Keep chart code close to the summary table it visualizes.
- In Colab, use explicit output paths because the runtime is temporary.
- In local mode, use repository output folders so exports stay versionable and easy to inspect.

Common pitfalls:
- Plotting too many categories in one chart.
- Using pie charts when bars or lines explain the comparison more clearly.
- Mixing incompatible scales without explanation.
- Spending too much time styling before the underlying summary is correct.
- Showing a chart without also checking the table behind it.
- Forgetting that the plotting stage can reveal mistakes in earlier cleaning or grouping logic.

Documentation references:
- [pandas chart visualization guide](https://pandas.pydata.org/docs/user_guide/visualization.html)
- [pandas.DataFrame.plot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html)
- [Matplotlib plot types](https://matplotlib.org/stable/plot_types/index.html)
- [Matplotlib annotated heatmap example](https://matplotlib.org/stable/gallery/images_contours_and_fields/image_annotated_heatmap.html)
- [Matplotlib subplots guide](https://matplotlib.org/stable/gallery/subplots_axes_and_figures/subplots_demo.html)
- [Seaborn plotting function overview](https://seaborn.pydata.org/tutorial/function_overview.html)


In [ ]:
monthly_totals = region_target_summary.groupby("month")[["revenue", "target_revenue"]].sum().round(2)
category_totals = (
    sales_analysis.groupby("category")["net_revenue"].sum().sort_values(ascending=False).round(2)
)
channel_month = (
    sales_analysis.groupby(["month", "sales_channel"])["net_revenue"]
    .sum()
    .unstack(fill_value=0)
    .round(2)
)
achievement_heatmap = (
    region_target_summary.pivot(index="region", columns="month", values="achievement_pct")
    .fillna(0)
    .round(1)
)

category_colors = {
    "Electronics": "#4C78A8",
    "Office": "#F58518",
    "Breakroom": "#54A24B",
}

fig1, axes1 = plt.subplots(2, 2, figsize=(15, 10))

monthly_totals.plot(kind="bar", ax=axes1[0, 0], title="Actual Revenue vs Target by Month")
axes1[0, 0].set_xlabel("Month")
axes1[0, 0].set_ylabel("Revenue")
axes1[0, 0].tick_params(axis="x", rotation=0)
axes1[0, 0].grid(axis="y", alpha=0.25)

category_totals.plot(
    kind="bar",
    ax=axes1[0, 1],
    color=[category_colors.get(category, "#999999") for category in category_totals.index],
    title="Revenue by Product Category",
)
axes1[0, 1].set_xlabel("Category")
axes1[0, 1].set_ylabel("Revenue")
axes1[0, 1].tick_params(axis="x", rotation=20)
axes1[0, 1].grid(axis="y", alpha=0.25)
for patch in axes1[0, 1].patches:
    height = patch.get_height()
    axes1[0, 1].annotate(
        f"{height:.0f}",
        (patch.get_x() + patch.get_width() / 2, height),
        ha="center",
        va="bottom",
        xytext=(0, 4),
        textcoords="offset points",
        fontsize=9,
    )

channel_month.plot(
    kind="line",
    marker="o",
    linewidth=2,
    ax=axes1[1, 0],
    title="Monthly Revenue by Sales Channel",
)
axes1[1, 0].set_xlabel("Month")
axes1[1, 0].set_ylabel("Revenue")
axes1[1, 0].tick_params(axis="x", rotation=0)
axes1[1, 0].grid(axis="y", alpha=0.25)

sales_analysis.boxplot(column="net_revenue", by="sales_channel", ax=axes1[1, 1])
axes1[1, 1].set_title("Order Revenue Distribution by Channel")
axes1[1, 1].set_xlabel("Sales Channel")
axes1[1, 1].set_ylabel("Net Revenue")
axes1[1, 1].grid(axis="y", alpha=0.25)
fig1.suptitle("")

plt.tight_layout()
dashboard_path = OUTPUT_DIR / "day4_visualization_dashboard.png"
fig1.savefig(dashboard_path, dpi=150, bbox_inches="tight")
plt.show()

fig2, axes2 = plt.subplots(1, 2, figsize=(15, 5.5))

for category, subset in sales_analysis.groupby("category"):
    axes2[0].scatter(
        subset["units"],
        subset["net_revenue"],
        s=90,
        alpha=0.8,
        label=category,
        color=category_colors.get(category, "#999999"),
    )
axes2[0].set_title("Units vs Net Revenue")
axes2[0].set_xlabel("Units")
axes2[0].set_ylabel("Net Revenue")
axes2[0].grid(alpha=0.25)
axes2[0].legend(title="Category")

heatmap = axes2[1].imshow(achievement_heatmap.values, aspect="auto", cmap="YlGnBu")
axes2[1].set_title("Target Achievement Heatmap (%)")
axes2[1].set_xticks(range(len(achievement_heatmap.columns)))
axes2[1].set_xticklabels(achievement_heatmap.columns)
axes2[1].set_yticks(range(len(achievement_heatmap.index)))
axes2[1].set_yticklabels(achievement_heatmap.index)
for row_index in range(len(achievement_heatmap.index)):
    for col_index in range(len(achievement_heatmap.columns)):
        value = achievement_heatmap.iloc[row_index, col_index]
        axes2[1].text(
            col_index,
            row_index,
            f"{value:.0f}",
            ha="center",
            va="center",
            color="black",
            fontsize=9,
        )
fig2.colorbar(heatmap, ax=axes2[1], label="Achievement %")

plt.tight_layout()
examples_path = OUTPUT_DIR / "day4_visualization_examples.png"
fig2.savefig(examples_path, dpi=150, bbox_inches="tight")
plt.show()

visualization_outputs = pd.DataFrame(
    [
        {"artifact": "visual dashboard", "path": str(dashboard_path)},
        {"artifact": "extra examples", "path": str(examples_path)},
    ]
)
visualization_outputs


## 9. Export

Goal: deliver outputs that other people and later scripts can reuse.

Export is the point where a notebook stops being a private exploration and becomes part of a repeatable workflow. Good export choices depend on the audience. A colleague may want Excel, a pipeline may want parquet, a web service may want JSON, and a reporting layer may want a styled worksheet or HTML table.

Common export targets:
- `to_csv()` for universal exchange and simple downstream use.
- `to_excel()` for spreadsheet consumers and multi-sheet outputs.
- `to_parquet()` for efficient analytics storage and type preservation.
- `to_json()` for web-friendly interchange.
- `to_html()` for lightweight publishing.
- `to_sql()` for loading processed results back into a database.

Export decisions worth making explicit:
- Should the index be saved or reset first?
- Should the output contain cleaned detail rows, summary tables, or both?
- Are there multiple deliverables such as `cleaned`, `summary`, and `chart_ready` versions?
- Does the file name include a date, version, or workflow stage?
- Is compression useful for large text outputs?

Reproducibility habits:
- Export from named final DataFrames rather than from intermediate temporary objects.
- Keep output paths predictable and separate from raw input paths.
- Save summaries in a sorted and presentation-ready order.
- If the same export is reused often, turn it into a helper cell or function later.

Common pitfalls:
- Accidentally exporting the index as an unnamed extra column.
- Exporting wide reports that are hard to reuse programmatically when a long clean table should also be saved.
- Overwriting prior outputs without a naming convention.
- Delivering only a chart when the underlying summary table is also needed.

Documentation references:
- [pandas IO tools user guide](https://pandas.pydata.org/docs/user_guide/io.html)
- [pandas.DataFrame.to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)
- [pandas.DataFrame.to_excel](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_excel.html)
- [pandas.DataFrame.to_parquet](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html)


In [ ]:
clean_output_path = OUTPUT_DIR / "sales_analysis_clean.csv"
summary_output_path = OUTPUT_DIR / "regional_target_summary.csv"
metrics_output_path = OUTPUT_DIR / "regional_metrics_long.json"
html_output_path = OUTPUT_DIR / "regional_revenue_report.html"

sales_analysis.to_csv(clean_output_path, index=False)
region_target_summary.to_csv(summary_output_path, index=False)
metrics_long.to_json(metrics_output_path, orient="records", indent=2)
report_wide.to_html(html_output_path)

if EXCEL_ENGINE:
    excel_output_path = OUTPUT_DIR / "day4_reporting_pack.xlsx"
    with pd.ExcelWriter(excel_output_path, engine=EXCEL_ENGINE) as writer:
        sales_analysis.to_excel(writer, index=False, sheet_name="sales_analysis")
        region_target_summary.to_excel(writer, index=False, sheet_name="region_targets")
        report_wide.reset_index().to_excel(writer, index=False, sheet_name="revenue_wide")
else:
    excel_output_path = None

artifact_rows = [
    {"artifact": "clean sales data", "path": str(clean_output_path)},
    {"artifact": "regional target summary", "path": str(summary_output_path)},
    {"artifact": "long metrics json", "path": str(metrics_output_path)},
    {"artifact": "html report", "path": str(html_output_path)},
    {"artifact": "excel pack", "path": str(excel_output_path) if excel_output_path else "skipped: no Excel engine installed"},
]

if "visualization_outputs" in globals():
    artifact_rows.extend(visualization_outputs.to_dict(orient="records"))

pd.DataFrame(artifact_rows)


## End-to-End Checklist

Before calling a pandas workflow complete, verify the following:
- The source loading choices are explicit and reproducible.
- The table grain is known and written down.
- Data types and missing values have been reviewed, not guessed.
- Cleaning rules are tied to observed issues.
- Filters reflect the analysis question and are readable.
- Merges and concatenations were validated with row counts and key checks.
- Summaries can be traced back to the source rows.
- Reshaping decisions support the reporting or visualization goal.
- Visuals answer a question clearly and are based on trustworthy tables.
- Exports are saved in formats that suit both humans and downstream code.


## Conclusion: Effective Workflow

An effective pandas workflow is not about memorizing isolated commands. It is about moving through a disciplined sequence where each stage prepares the next one: load carefully, inspect honestly, clean explicitly, filter intentionally, combine cautiously, summarize meaningfully, reshape for the task, visualize with purpose, and export for reuse.

When this workflow is done well, the notebook becomes more than a place to test code. It becomes a reproducible record of how raw information was turned into reliable analysis. That is the real value of pandas in practical work.
